# Tiền xử lý dữ liệu dự báo Top 50 theo thị trường
Notebook nạp và làm sạch dữ liệu Spotify, ghép dữ liệu quốc gia rồi tạo các bảng phục vụ phân tích. Phần dự báo dùng một mẫu cho mỗi bài trong từng thị trường, gồm cả Global, để xem bài có vào Top 50 trong tám tuần tiếp theo hay không.

## Phần 1: Khởi tạo và nạp dữ liệu
Chuẩn bị thư viện, đường dẫn và các cột dùng trong quy trình. Sau khi nạp chart, quan sát dữ liệu trước khi làm sạch.

### Khởi tạo môi trường và nạp thư viện
Nạp các thư viện cần dùng và xác định đường dẫn đến dữ liệu. Kết quả được lưu trong thư mục processed_market_forecast để dễ phân biệt với quy trình cũ.

In [1]:
from pathlib import Path
import sys
import json
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display
from importlib.metadata import version
project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
path_raw = project_root / 'data' / 'raw'
path_processed = project_root / 'data' / 'processed_market_forecast'
path_charts = path_raw / 'spotify-weekly-charts.csv'
path_songs = path_raw / 'songs.csv'
sys.path.insert(0, str(project_root)) if str(project_root) not in sys.path else None
from src.data_loader import COLUMNS_TO_KEEP, AUDIO_FEATURE_COLS, COUNTRY_ALIAS_MAP
assert path_charts.is_file()
print(f'thư mục đầu ra riêng ({path_processed}):')
pd.set_option("display.max_columns", None)

thư mục đầu ra riêng (D:\uni\hk1nam3\idvi\idvi-final_project\Spotify_Audio_Trends_Analytics\data\processed_market_forecast):


### Khai báo các cột và quy tắc dự báo
Mỗi mẫu là một bài hát trong một thị trường, tại tuần đầu thấy bài đó trong dữ liệu. Mục tiêu là dự báo bài có vào Top 50 của thị trường đó trong tám tuần tiếp theo hay không.

In [2]:
audio_cols = list(AUDIO_FEATURE_COLS)
unit_audio_cols = ['danceability', 'energy', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence']
slot_grain_cols = ['chart_country', 'week', 'rank']
chart_grain_cols = ['track_id', 'artist_id', 'chart_country', 'week']
track_week_cols = ['track_id', 'week']
pair_cols = ['track_id', 'chart_country']
forecast_audio_cols = [column for column in audio_cols if column not in ['key', 'mode']]
feature_cols = [column for column in forecast_audio_cols if column != 'duration'] + ['duration_min', 'rank_first_week', 'log_streams_first_week', 'chart_country']
publication_lag_days_proxy = 4
minimum_country_observed_weeks = 75
runtime_versions = {name: version(name) for name in ['pandas', 'numpy', 'pyarrow', 'nbformat']}
df_contract = pd.DataFrame({'item': ['sample', 'target', 'features', 'time'],
    'rule': ['track_id + chart_country tại first observed', 'Top 50 cùng thị trường trong t0+7 đến t0+56 ngày',
             'audio + rank + log streams tuần đầu + thị trường', 'timestamp công bố chưa biết; proxy t0+4 ngày được đánh dấu']})
display(df_contract)

,item,rule
0,sample,track_id + chart_country tại first observed
1,target,Top 50 cùng thị trường trong t0+7 đến t0+56 ngày
2,features,audio + rank + log streams tuần đầu + thị trường
3,time,timestamp công bố chưa biết; proxy t0+4 ngày đ...


### Ghi thông tin nguồn dữ liệu
Lưu tên nguồn và những thông tin đã biết của từng file. Ngày tải và giấy phép chưa được xác minh sẽ để trống; nguồn hướng dẫn về tuần chart được ghi kèm trong bảng.

In [3]:
source_manifest = {
    'spotify-weekly-charts.csv': {'source': 'Kaggle yelexa/spotify200', 'week_definition': 'Thursday dates consistent with official week end; file convention provisional'},
    'songs.csv': {'source': 'candidate Kaggle serkantysz/550k-spotify-songs-audio-lyrics-and-genres', 'forecast_use': 'excluded: no historical popularity timestamp'},
    'country_codes.csv': {'source': 'World Bank metadata via local Kaggle snapshot', 'vintage': None},
    'gdp_data.csv': {'source': 'World Bank via local Kaggle snapshot', 'indicator_unit': 'unverified'},
    'whs_years_updated.csv': {'source': 'World Happiness Report merged local snapshot'},
    'global_deaths.csv': {'source': 'JHU CSSE convenient daily deaths snapshot'}}
for source_info in source_manifest.values():
    source_info.update(downloaded_at=None, snapshot_version=None, snapshot_license=None)
source_manifest['spotify-weekly-charts.csv']['official_reference'] = 'https://support.spotify.com/dm/artists/article/understanding-spotify-charts/'
source_manifest['spotify-weekly-charts.csv']['prediction_time_proxy_lag_days'] = publication_lag_days_proxy
display(pd.DataFrame(source_manifest).T)

,source,week_definition,downloaded_at,snapshot_version,snapshot_license,official_reference,prediction_time_proxy_lag_days,forecast_use,vintage,indicator_unit
spotify-weekly-charts.csv,Kaggle yelexa/spotify200,Thursday dates consistent with official week e...,None,None,None,https://support.spotify.com/dm/artists/article...,4,NaN,NaN,NaN
songs.csv,candidate Kaggle serkantysz/550k-spotify-songs...,NaN,NaN,NaN,NaN,NaN,NaN,excluded: no historical popularity timestamp,NaN,NaN
country_codes.csv,World Bank metadata via local Kaggle snapshot,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gdp_data.csv,World Bank via local Kaggle snapshot,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unverified
whs_years_updated.csv,World Happiness Report merged local snapshot,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
global_deaths.csv,JHU CSSE convenient daily deaths snapshot,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Nạp dữ liệu bảng xếp hạng
Đọc file chart theo từng phần để giảm lượng bộ nhớ cần dùng. Những dòng tiêu đề bị lặp trong file được bỏ đi và số dòng được kiểm tra lại.

In [4]:
chart_columns = COLUMNS_TO_KEEP['spotify-weekly-charts.csv']['keep']
chart_chunks = []
raw_row_count = 0
header_row_count = 0
for df_chunk in pd.read_csv(path_charts, usecols=chart_columns, chunksize=300_000, low_memory=False):
    raw_row_count += len(df_chunk)
    header_mask = df_chunk['rank'].astype(str).eq('rank') | df_chunk['streams'].astype(str).eq('streams')
    header_row_count += int(header_mask.sum())
    df_chunk_clean = df_chunk.loc[~header_mask].copy()
    assert len(df_chunk_clean) == len(df_chunk) - int(header_mask.sum())
    chart_chunks.append(df_chunk_clean)
df_charts_raw = pd.concat(chart_chunks, ignore_index=True)
assert len(df_charts_raw) == raw_row_count - header_row_count
del chart_chunks, df_chunk, df_chunk_clean
print(f'số dòng thô ({raw_row_count:,}) và sau loại header ({len(df_charts_raw):,}):')
print(f'số header lặp ({header_row_count:,}):')
display(df_charts_raw.head(3))

số dòng thô (1,787,999) và sau loại header (1,787,926):
số header lặp (73):


,uri,rank,artist_names,artists_num,artist_individual,artist_id,artist_genre,collab,track_name,release_date,peak_rank,previous_rank,weeks_on_chart,streams,week,danceability,energy,key,mode,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration,country,region,language
0,spotify:track:2gpQi3hbcUAcEG8m2dlgfB,1,Paulo Londra,1.0,Paulo Londra,spotify:artist:3vQ0GE3mI0dAaxIMYe5g7z,argentine hip hop,0,Plan A,2022-03-23,1,1,4,3003411,2022-04-14,0.583,0.8340000000000001,0.0,1.0,-4.875,0.0444,0.0495,0.0,0.0658,0.557,173.935,178203.0,Argentina,South America,Spanish
1,spotify:track:2x8oBuYaObjqHqgGuIUZ0b,2,WOS,1.0,WOS,spotify:artist:5YCc6xS5Gpj3EkaYGdjyNK,argentine indie,0,ARRANCARMELO,2022-04-06,2,129,2,2512175,2022-04-14,0.654,0.354,5.0,1.0,-7.358,0.0738,0.7240000000000001,0.0,0.134,0.262,81.956,183547.0,Argentina,South America,Spanish
2,spotify:track:2SJZdZ5DLtlRosJ2xHJJJa,3,Paulo Londra,1.0,Paulo Londra,spotify:artist:3vQ0GE3mI0dAaxIMYe5g7z,argentine hip hop,0,Chance,2022-04-06,3,59,2,2408983,2022-04-14,0.721,0.463,1.0,0.0,-9.483,0.0646,0.241,0.0,0.0929,0.21600000000000005,137.915,204003.0,Argentina,South America,Spanish


### Quan sát những dòng đầu và cuối
Xem vài dòng ở đầu và cuối bảng để nắm cấu trúc dữ liệu. Bước này cũng giúp kiểm tra file đã được đọc đến cuối.

In [5]:
print(f'ba dòng đầu:')
display(df_charts_raw.head(3))
print(f'ba dòng cuối:')
display(df_charts_raw.tail(3))

ba dòng đầu:


,uri,rank,artist_names,artists_num,artist_individual,artist_id,artist_genre,collab,track_name,release_date,peak_rank,previous_rank,weeks_on_chart,streams,week,danceability,energy,key,mode,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration,country,region,language
0,spotify:track:2gpQi3hbcUAcEG8m2dlgfB,1,Paulo Londra,1.0,Paulo Londra,spotify:artist:3vQ0GE3mI0dAaxIMYe5g7z,argentine hip hop,0,Plan A,2022-03-23,1,1,4,3003411,2022-04-14,0.583,0.8340000000000001,0.0,1.0,-4.875,0.0444,0.0495,0.0,0.0658,0.557,173.935,178203.0,Argentina,South America,Spanish
1,spotify:track:2x8oBuYaObjqHqgGuIUZ0b,2,WOS,1.0,WOS,spotify:artist:5YCc6xS5Gpj3EkaYGdjyNK,argentine indie,0,ARRANCARMELO,2022-04-06,2,129,2,2512175,2022-04-14,0.654,0.354,5.0,1.0,-7.358,0.0738,0.7240000000000001,0.0,0.134,0.262,81.956,183547.0,Argentina,South America,Spanish
2,spotify:track:2SJZdZ5DLtlRosJ2xHJJJa,3,Paulo Londra,1.0,Paulo Londra,spotify:artist:3vQ0GE3mI0dAaxIMYe5g7z,argentine hip hop,0,Chance,2022-04-06,3,59,2,2408983,2022-04-14,0.721,0.463,1.0,0.0,-9.483,0.0646,0.241,0.0,0.0929,0.21600000000000005,137.915,204003.0,Argentina,South America,Spanish


ba dòng cuối:


,uri,rank,artist_names,artists_num,artist_individual,artist_id,artist_genre,collab,track_name,release_date,peak_rank,previous_rank,weeks_on_chart,streams,week,danceability,energy,key,mode,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration,country,region,language
1787923,spotify:track:6JIC3hbC28JZKZ8AlAqX8h,199,"Elton John, Dua Lipa, PNAU",3.0,Elton John,spotify:artist:3PhoLpVuITZKcymswpck5b,mellow gold,1,Cold Heart - PNAU Remix,2021-10-22,114,183,30,36817,2022-05-05,0.795,0.8,1,1,-6.32,0.0309,0.0354,7.25e-05,0.0915,0.934,116.032,202735,Vietnam,Asia,Vietnamese
1787924,spotify:track:6JIC3hbC28JZKZ8AlAqX8h,199,"Elton John, Dua Lipa, PNAU",3.0,Dua Lipa,spotify:artist:6M2wZ9GZgrQXHCFfjv46we,pop,1,Cold Heart - PNAU Remix,2021-10-22,114,183,30,36817,2022-05-05,0.795,0.8,1,1,-6.32,0.0309,0.0354,7.25e-05,0.0915,0.934,116.032,202735,Vietnam,Asia,Vietnamese
1787925,spotify:track:6JIC3hbC28JZKZ8AlAqX8h,199,"Elton John, Dua Lipa, PNAU",3.0,PNAU,spotify:artist:6n28c9qs9hNGriNa72b26u,alternative dance,1,Cold Heart - PNAU Remix,2021-10-22,114,183,30,36817,2022-05-05,0.795,0.8,1,1,-6.32,0.0309,0.0354,7.25e-05,0.0915,0.934,116.032,202735,Vietnam,Asia,Vietnamese


### Lấy mẫu và kiểm tra kiểu dữ liệu
Xem ba dòng ngẫu nhiên và thống kê kiểu dữ liệu, số giá trị thiếu. Kết quả cho biết những cột cần xử lý ở phần tiếp theo.

In [6]:
assert len(df_charts_raw) >= 3
print(f'ba dòng ngẫu nhiên:')
display(df_charts_raw.sample(3, random_state=42))
df_raw_columns = pd.DataFrame({'dtype': df_charts_raw.dtypes.astype(str), 'missing_count': df_charts_raw.isna().sum()})
print(f'kiểu dữ liệu và số giá trị thiếu:')
display(df_raw_columns)

ba dòng ngẫu nhiên:


,uri,rank,artist_names,artists_num,artist_individual,artist_id,artist_genre,collab,track_name,release_date,peak_rank,previous_rank,weeks_on_chart,streams,week,danceability,energy,key,mode,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration,country,region,language
1701836,spotify:track:04gmklgYhNhZxRnAvSofoE,111,"pyrokinesis, MUKKA",2.0,MUKKA,spotify:artist:6a1CGYBFAUrrWZvD2iSsEi,russian alt pop,1,Днями-ночами,2020-03-19,111,133,33,33416,2021-11-11,0.564,0.878,9,1,-1.592,0.0556,0.179,0.0,0.263,0.313,179.815,197333,Ukraine,Europe,Ukrainian
1306232,spotify:track:2mM3gZ0BbPwPPMelbA8vgt,10,"Milly, Farruko, Nio Garcia, Jay Wheeler, Amenazzy",5.0,Milly,spotify:artist:7kXCcNRLEnblbTAppPraBp,reggaeton flow,1,No Te Enamores - Remix,2020-11-20,8,9,11,524204,2021-03-11,0.8240000000000001,0.8370000000000001,5,1,-4.603,0.107,0.573,1.66e-05,0.271,0.58,100.021,289933,Peru,South America,Spanish
450242,spotify:track:5wANPM4fQCJwkGd4rN57mH,78,Olivia Rodrigo,1.0,Olivia Rodrigo,spotify:artist:1McMsnEElThX1knmY4oliG,pop,0,drivers license,2021-05-21,22,78,28,27820,2021-07-22,0.561,0.431,10,1,-8.81,0.0578,0.768,1.42e-05,0.106,0.13699999999999998,143.875,242013,El Salvador,Central America,Spanish


kiểu dữ liệu và số giá trị thiếu:


,dtype,missing_count
uri,str,0
rank,str,0
artist_names,str,0
artists_num,str,0
artist_individual,str,0
artist_id,str,0
artist_genre,str,0
collab,str,0
track_name,str,0
release_date,str,0


### Kiểm tra dòng trùng và số bài trong bảng thô
Đếm dòng trùng hoàn toàn và số mã bài khác nhau trước khi làm sạch. Nhiều dòng của một bài có thể đến từ nhiều nghệ sĩ, tuần hoặc thị trường.

In [7]:
raw_duplicate_count = int(df_charts_raw.duplicated().sum())
raw_track_count = df_charts_raw['uri'].nunique()
print(f'dòng trùng hoàn toàn ({raw_duplicate_count:,}):')
print(f'số mã bài trong bảng thô ({raw_track_count:,}):')

dòng trùng hoàn toàn (0):
số mã bài trong bảng thô (39,202):


## Phần 2: Làm sạch bảng xếp hạng
Chuẩn hóa kiểu dữ liệu, mã định danh, thời gian và các chỉ số âm thanh. Lịch sử thứ hạng được giữ trước khi lọc âm thanh để không làm mất thông tin dùng tạo nhãn.

### Chuyển các cột số về đúng kiểu
Tạo bảng làm sạch rồi chuyển các cột thứ hạng, lượt nghe và âm thanh sang kiểu số. Giá trị không chuyển được được ghi lại để xử lý ở bước sau.

In [8]:
numeric_columns = ['rank', 'peak_rank', 'weeks_on_chart', 'streams', 'artists_num'] + audio_cols
numeric_audit = []
df_charts_clean = df_charts_raw.copy()
for column in numeric_columns:
    source_missing = df_charts_clean[column].isna()
    converted_values = pd.to_numeric(df_charts_clean[column], errors='coerce')
    numeric_audit.append({'column': column, 'source_missing': int(source_missing.sum()),
                          'invalid_numeric': int((~source_missing & converted_values.isna()).sum())})
    df_charts_clean[column] = converted_values
df_numeric_audit = pd.DataFrame(numeric_audit)
del df_charts_raw
print(f'kết quả chuyển kiểu số:')
display(df_numeric_audit)

kết quả chuyển kiểu số:


,column,source_missing,invalid_numeric
0,rank,0,0
1,peak_rank,0,0
2,weeks_on_chart,0,0
3,streams,0,0
4,artists_num,0,0
5,danceability,395,0
6,energy,395,0
7,key,395,0
8,mode,395,0
9,loudness,395,0


### Thống kê các cột số sau khi chuyển kiểu
Xem khoảng giá trị và mức phân tán của các cột số. Bảng thống kê giúp nhận biết các giá trị cần được kiểm tra giới hạn.

In [9]:
print(f'thống kê các cột số:')
display(df_charts_clean[numeric_columns].describe().T)

thống kê các cột số:


,count,mean,std,min,25%,50%,75%,max
rank,1787926.0,99.068727,5.773889e+01,1.000000,49.0000,98.0000,149.000000,2.000000e+02
peak_rank,1787926.0,40.074916,4.561516e+01,1.000000,5.0000,21.0000,61.000000,2.000000e+02
weeks_on_chart,1787926.0,35.287830,4.266578e+01,1.000000,7.0000,20.0000,46.000000,2.900000e+02
streams,1787926.0,697469.255770,2.381627e+06,2067.000000,24859.0000,64732.0000,314269.000000,8.495293e+07
artists_num,1787926.0,2.244799,1.610825e+00,1.000000,1.0000,2.0000,3.000000,2.000000e+01
danceability,1787531.0,0.698214,1.319124e-01,0.085900,0.6130,0.7210,0.797000,9.860000e-01
energy,1787531.0,0.656129,1.562644e-01,0.003160,0.5570,0.6730,0.775000,1.000000e+00
key,1787531.0,5.273468,3.616229e+00,0.000000,2.0000,5.0000,8.000000,1.100000e+01
mode,1787531.0,0.551461,4.973449e-01,0.000000,0.0000,1.0000,1.000000,1.000000e+00
loudness,1787531.0,-6.175125,2.386746e+00,-41.410000,-7.5080,-5.8370,-4.516000,1.906000e+00


### Chuẩn hóa mã bài và mã nghệ sĩ
Lấy phần mã Spotify từ chuỗi định danh của bài và nghệ sĩ. Các mã này được dùng làm khóa để kiểm tra và ghép bảng.

In [10]:
df_charts_clean['track_id'] = df_charts_clean['uri'].astype('str').str.split(':').str[-1]
df_charts_clean['artist_id'] = df_charts_clean['artist_id'].astype('str').str.split(':').str[-1]
print(f'mã bài và mã nghệ sĩ sau chuẩn hóa:')
display(df_charts_clean[['track_id', 'artist_id']].head(3))

mã bài và mã nghệ sĩ sau chuẩn hóa:


,track_id,artist_id
0,2gpQi3hbcUAcEG8m2dlgfB,3vQ0GE3mI0dAaxIMYe5g7z
1,2x8oBuYaObjqHqgGuIUZ0b,5YCc6xS5Gpj3EkaYGdjyNK
2,2SJZdZ5DLtlRosJ2xHJJJa,3vQ0GE3mI0dAaxIMYe5g7z


### Chuẩn hóa tuần và lấy năm, tháng
Chuyển tuần chart sang kiểu ngày tháng để tính khoảng thời gian. Lấy thêm năm và tháng để dùng khi tổng hợp dữ liệu.

In [11]:
df_charts_clean['week'] = pd.to_datetime(df_charts_clean['week'], errors='coerce')
df_charts_clean['year'] = df_charts_clean['week'].dt.year
df_charts_clean['month'] = df_charts_clean['week'].dt.month
print(f'tuần, năm và tháng:')
display(df_charts_clean[['week', 'year', 'month']].head(3))

tuần, năm và tháng:


,week,year,month
0,2022-04-14,2022,4
1,2022-04-14,2022,4
2,2022-04-14,2022,4


### Bổ sung các cột thị trường
Giữ các cột gốc và tạo thêm cột có tiền tố chart. Cách đặt tên này giúp phân biệt thông tin chart với dữ liệu quốc gia ghép vào sau.

In [12]:
df_charts_clean['chart_country'] = df_charts_clean['country']
df_charts_clean['chart_region'] = df_charts_clean['region']
df_charts_clean['chart_language'] = df_charts_clean['language']
print(f'thông tin thị trường:')
display(df_charts_clean[['chart_country', 'chart_region', 'chart_language']].head(3))

thông tin thị trường:


,chart_country,chart_region,chart_language
0,Argentina,South America,Spanish
1,Argentina,South America,Spanish
2,Argentina,South America,Spanish


### Giữ lịch sử xếp hạng trước khi lọc âm thanh
Kiểm tra mã bài, thị trường, ngày và thứ hạng để giữ lại lịch sử chart hợp lệ. Bước này làm trước khi lọc âm thanh, vì một dòng thiếu âm thanh vẫn có thể cho biết bài đã vào Top 50.

In [13]:
history_invalid = (~df_charts_clean['track_id'].str.fullmatch(r'[A-Za-z0-9]{22}', na=False)
    | df_charts_clean['chart_country'].isna() | df_charts_clean['week'].isna()
    | df_charts_clean['week'].dt.weekday.ne(3) | ~df_charts_clean['rank'].between(1, 200)
    | df_charts_clean['rank'].mod(1).ne(0))
# Giữ lịch sử thứ hạng trước khi lọc lỗi âm thanh.
df_history_invalid_rows = df_charts_clean.loc[history_invalid].copy()
df_history_rows = df_charts_clean.loc[~history_invalid].copy()
assert len(df_history_rows) + len(df_history_invalid_rows) == len(df_charts_clean)
streams_valid = np.isfinite(df_history_rows['streams']) & df_history_rows['streams'].ge(0) & df_history_rows['streams'].mod(1).eq(0)
df_history_stream_errors = df_history_rows.loc[~streams_valid, ['track_id', 'chart_country', 'week', 'rank', 'streams']].copy()
assert len(df_history_stream_errors) == int((~streams_valid).sum())
df_history_rows['streams'] = df_history_rows['streams'].where(streams_valid).astype('Int64')
df_history_rows['rank'] = df_history_rows['rank'].astype('int16')
print(f'lịch trước và sau lọc cấu trúc ({len(df_charts_clean):,} → {len(df_history_rows):,}):')

lịch trước và sau lọc cấu trúc (1,787,926 → 1,787,926):


### Kiểm tra các dòng nghệ sĩ trong cùng vị trí
Một bài có nhiều nghệ sĩ có thể được ghi thành nhiều dòng. Các dòng cùng thị trường, tuần và thứ hạng phải có cùng mã bài và lượt nghe; nhóm không khớp được lưu riêng.

In [14]:
df_history_slot_checks = df_history_rows.groupby(slot_grain_cols, observed=True)[['track_id', 'streams']].nunique(dropna=False)
conflict_mask = df_history_slot_checks.gt(1).any(axis=1)
df_history_slot_conflicts = df_history_slot_checks.loc[conflict_mask].reset_index()
assert len(df_history_slot_conflicts) == int(conflict_mask.sum())
df_history_rows = df_history_rows.merge(df_history_slot_conflicts[slot_grain_cols], on=slot_grain_cols,
    how='left', validate='many_to_one', indicator='slot_conflict_join')
assert len(df_history_rows) + len(df_history_invalid_rows) == len(df_charts_clean)
df_history_conflict_rows = df_history_rows.loc[df_history_rows['slot_conflict_join'].eq('both')].copy()
df_history_consistent = df_history_rows.loc[df_history_rows['slot_conflict_join'].eq('left_only')].copy()
assert len(df_history_consistent) + len(df_history_conflict_rows) == len(df_history_rows)
print(f'lịch trước và sau cách ly vị trí ({len(df_history_rows):,} → {len(df_history_consistent):,}):')

lịch trước và sau cách ly vị trí (1,787,926 → 1,787,926):


### Giữ một dòng cho mỗi vị trí xếp hạng
Bỏ các dòng lặp do nhiều nghệ sĩ sau khi đã kiểm tra chúng khớp nhau. Đối chiếu tổng lượt nghe để chắc chắn việc bỏ dòng không làm sai số liệu.

In [15]:
# Mỗi vị trí chỉ tính lượt nghe một lần.
df_slots_observed = df_history_consistent.drop_duplicates(slot_grain_cols).copy()
assert not df_slots_observed.duplicated(slot_grain_cols).any()
assert not df_slots_observed.duplicated(pair_cols + ['week']).any()
assert df_slots_observed['streams'].sum() == df_history_consistent.groupby(slot_grain_cols, observed=True)['streams'].first().sum()
print(f'dòng nghệ sĩ và vị trí quan sát ({len(df_history_consistent):,} → {len(df_slots_observed):,}):')
df_history_row_counts = pd.DataFrame({'step': ['raw', 'after_headers', 'history_structural', 'history_slots'],
    'rows': [raw_row_count, raw_row_count - header_row_count, len(df_history_consistent), len(df_slots_observed)]})
display(df_history_row_counts)
del df_history_rows, df_history_consistent, df_history_slot_checks

dòng nghệ sĩ và vị trí quan sát (1,787,926 → 1,109,520):


,step,rows
0,raw,1787999
1,after_headers,1787926
2,history_structural,1787926
3,history_slots,1109520


### Kiểm tra mã định danh và thứ hạng
Kiểm tra mã Spotify, các trường bắt buộc và giới hạn của thứ hạng, lượt nghe. Các lỗi được đánh dấu để tách riêng ở bước tiếp theo.

In [16]:
df_chart_errors = pd.DataFrame(index=df_charts_clean.index)
df_chart_errors['invalid_track_id'] = ~df_charts_clean['track_id'].str.fullmatch(r'[A-Za-z0-9]{22}', na=False)
df_chart_errors['invalid_artist_id'] = ~df_charts_clean['artist_id'].str.fullmatch(r'[A-Za-z0-9]{22}', na=False)
df_chart_errors['invalid_week'] = df_charts_clean['week'].isna() | df_charts_clean['week'].dt.dayofweek.ne(3)
df_chart_errors['missing_market'] = df_charts_clean['chart_country'].isna()
for column in ['rank', 'peak_rank']:
    values = df_charts_clean[column]
    df_chart_errors['invalid_' + column] = ~values.between(1, 200) | values.mod(1).ne(0)
for column in ['streams', 'weeks_on_chart', 'artists_num']:
    values = df_charts_clean[column]
    minimum = 0 if column == 'streams' else 1
    df_chart_errors['invalid_' + column] = ~np.isfinite(values) | values.lt(minimum) | values.mod(1).ne(0)
display(df_chart_errors.sum().to_frame('row_count'))

,row_count
invalid_track_id,0
invalid_artist_id,0
invalid_week,0
missing_market,0
invalid_rank,0
invalid_peak_rank,0
invalid_streams,0
invalid_weeks_on_chart,0
invalid_artists_num,0


### Kiểm tra giá trị đặc trưng âm thanh
Kiểm tra giá trị thiếu, vô hạn và giới hạn hợp lệ của các chỉ số âm thanh. Những lỗi rõ ràng được đánh dấu trước khi dùng dữ liệu để phân tích.

In [17]:
df_chart_errors['missing_audio'] = df_charts_clean[audio_cols].isna().any(axis=1)
df_chart_errors['infinite_audio'] = np.isinf(df_charts_clean[audio_cols]).any(axis=1)
df_chart_errors['invalid_unit_audio'] = (~df_charts_clean[unit_audio_cols].ge(0) | ~df_charts_clean[unit_audio_cols].le(1)).any(axis=1)
df_chart_errors['invalid_key'] = ~df_charts_clean['key'].between(-1, 11) | df_charts_clean['key'].mod(1).ne(0)
df_chart_errors['invalid_mode'] = ~df_charts_clean['mode'].isin([0, 1])
df_chart_errors['invalid_duration'] = df_charts_clean['duration'].le(0)
df_chart_errors['invalid_tempo'] = df_charts_clean['tempo'].lt(0)
print(f'lỗi cứng theo quy tắc, một dòng có thể vi phạm nhiều quy tắc:')
display(df_chart_errors.sum().to_frame('row_count'))

lỗi cứng theo quy tắc, một dòng có thể vi phạm nhiều quy tắc:


,row_count
invalid_track_id,0
invalid_artist_id,0
invalid_week,0
missing_market,0
invalid_rank,0
invalid_peak_rank,0
invalid_streams,0
invalid_weeks_on_chart,0
invalid_artists_num,0
missing_audio,395


### Tách riêng những dòng có lỗi
Giữ các dòng hợp lệ trong bảng làm sạch và lưu các dòng lỗi kèm lý do. Tổng số dòng ở hai bảng phải bằng số dòng trước khi lọc.

In [18]:
chart_before_filter = len(df_charts_clean)
chart_reject_mask = df_chart_errors.any(axis=1)
df_charts_rejected = df_charts_clean.loc[chart_reject_mask].copy()
df_charts_rejected['rejection_reason'] = df_chart_errors.loc[chart_reject_mask].apply(
    lambda row: ', '.join(row.index[row.to_numpy()]), axis=1)
df_charts_clean = df_charts_clean.loc[~chart_reject_mask].copy()
assert len(df_charts_clean) + len(df_charts_rejected) == chart_before_filter
print(f'số dòng trước lọc ({chart_before_filter:,}) và sau lọc ({len(df_charts_clean):,}):')
display(df_charts_rejected[['track_id', 'chart_country', 'week', 'rejection_reason']].head(3))

số dòng trước lọc (1,787,926) và sau lọc (1,787,531):


,track_id,chart_country,week,rejection_reason
6910,5saUoeceT2zau6ScmU77e6,Argentina,2022-04-14,"missing_audio, invalid_unit_audio, invalid_key..."
6911,5saUoeceT2zau6ScmU77e6,Argentina,2022-04-14,"missing_audio, invalid_unit_audio, invalid_key..."
6912,5saUoeceT2zau6ScmU77e6,Argentina,2022-04-14,"missing_audio, invalid_unit_audio, invalid_key..."


### Chuẩn hóa kiểu dữ liệu và hạng tuần trước
Chuyển các cột sang kiểu dữ liệu phù hợp và tạo cờ bài mới vào bảng. Hạng tuần trước bị thiếu không được tự coi là một bài mới.

In [19]:
for column in ['rank', 'peak_rank', 'weeks_on_chart', 'streams', 'artists_num']:
    df_charts_clean[column] = df_charts_clean[column].astype('int64')
for column in ['key', 'mode']:
    df_charts_clean[column] = df_charts_clean[column].astype('int8')
df_charts_clean['year'] = df_charts_clean['year'].astype('int16')
df_charts_clean['month'] = df_charts_clean['month'].astype('int8')
previous_values = pd.to_numeric(df_charts_clean['previous_rank'], errors='coerce')
valid_previous = previous_values.between(1, 200) & previous_values.mod(1).eq(0)
df_charts_clean['previous_rank_status'] = np.select(
    [previous_values.eq(-1), df_charts_clean['previous_rank'].isna(), valid_previous],
    ['new_entry', 'missing', 'observed'], default='invalid')
df_charts_clean['is_new_entry'] = previous_values.eq(-1)
df_charts_clean['previous_rank'] = previous_values.where(valid_previous).astype('Int16')
display(df_charts_clean['previous_rank_status'].value_counts().to_frame('row_count'))

,row_count
previous_rank_status,
observed,1620012
new_entry,167519


### Chuẩn hóa ngày phát hành
Phân biệt ngày phát hành đủ ngày, tháng, năm với giá trị chỉ có tháng hoặc năm. Chỉ tính số ngày đến khi vào chart nếu ngày phát hành đủ chính xác.

In [20]:
release_text = df_charts_clean['release_date'].astype('str').str.strip()
df_charts_clean['release_date_precision'] = np.select(
    [release_text.str.fullmatch(r'\d{4}-\d{2}-\d{2}', na=False),
     release_text.str.fullmatch(r'\d{4}-\d{2}', na=False),
     release_text.str.fullmatch(r'\d{4}', na=False)], ['day', 'month', 'year'], default='unknown')
df_charts_clean['release_date_clean'] = pd.to_datetime(release_text, format='mixed', errors='coerce')
df_charts_clean.loc[df_charts_clean['release_date_clean'].isna(), 'release_date_precision'] = 'unknown'
df_charts_clean['release_year'] = df_charts_clean['release_date_clean'].dt.year.astype('Int16')
df_charts_clean['days_to_chart'] = (df_charts_clean['week'] - df_charts_clean['release_date_clean']).dt.days
df_charts_clean['days_to_chart'] = df_charts_clean['days_to_chart'].where(df_charts_clean['release_date_precision'].eq('day')).astype('Int32')
df_charts_clean['release_after_chart'] = df_charts_clean['days_to_chart'].lt(0).fillna(False)
display(df_charts_clean['release_date_precision'].value_counts().to_frame('row_count'))

,row_count
release_date_precision,
day,1770168
year,17113
month,250


### Chuẩn hóa thể loại và đánh dấu dòng cần xem lại
Ghi thể loại thiếu là unknown và giữ thêm cờ cho trường hợp này. Những giá trị âm thanh cần xem lại vẫn được giữ nếu chưa đủ căn cứ để loại.

In [21]:
genre_values = df_charts_clean['artist_genre'].astype('str').str.strip().str.lower()
df_charts_clean['genre_missing'] = genre_values.isna() | genre_values.isin(['0', '0.0', '', 'none', 'nan', 'null', 'unknown'])
df_charts_clean['artist_genre'] = genre_values.mask(df_charts_clean['genre_missing'], 'unknown')
df_charts_clean['audio_review_flag'] = df_charts_clean['loudness'].gt(0) | df_charts_clean['duration'].gt(600_000)
review_mask = df_charts_clean['audio_review_flag'] | df_charts_clean['release_after_chart']
df_chart_review = df_charts_clean.loc[review_mask].copy()
assert len(df_chart_review) == int(review_mask.sum())
for column in ['chart_country', 'chart_region', 'chart_language', 'artist_genre']:
    df_charts_clean[column] = df_charts_clean[column].astype('category')
print(f'số dòng cần xem xét ({len(df_chart_review):,}) trong {len(df_charts_clean):,} dòng:')
display(df_chart_review[['track_id', 'week', 'release_after_chart', 'audio_review_flag']].head(3))

số dòng cần xem xét (4,824) trong 1,787,531 dòng:


,track_id,week,release_after_chart,audio_review_flag
244,1ba1ZedHHXyiQdFQM2Baak,2021-04-08,False,True
248,4a0XLZUjHFHPwMBxQJD2JX,2021-04-08,False,True
303,3fq0e1TpOQFBDi0vuYrePo,2021-06-24,False,True


## Phần 3: Kiểm tra bài hát và các dòng nghệ sĩ
Kiểm tra khóa, bỏ các dòng nghệ sĩ lặp khi tính lượt nghe và tạo bảng bài hát. Những trường hợp chưa thống nhất được lưu riêng để xem lại.

### Kiểm tra khóa và tách Global với quốc gia
Kiểm tra mỗi dòng theo mã bài, nghệ sĩ, thị trường và tuần. Sau đó tách bảng Global và bảng quốc gia, đồng thời đối chiếu số dòng.

In [22]:
assert df_charts_clean[chart_grain_cols].notna().all().all()
assert not df_charts_clean.duplicated(chart_grain_cols).any()
global_mask = df_charts_clean['chart_country'].eq('Global')
df_charts_global = df_charts_clean.loc[global_mask].copy()
df_charts_country = df_charts_clean.loc[~global_mask].copy()
assert len(df_charts_global) + len(df_charts_country) == len(df_charts_clean)
print(f'số dòng trước tách ({len(df_charts_clean):,}), global ({len(df_charts_global):,}), quốc gia ({len(df_charts_country):,}):')
display(df_charts_clean.groupby('chart_country', observed=True)['week'].agg(['min', 'max', 'nunique']))

số dòng trước tách (1,787,531), global (95,932), quốc gia (1,691,599):


,min,max,nunique
chart_country,,,
Argentina,2021-02-04,2022-07-14,76
Australia,2021-02-04,2022-07-14,76
Austria,2021-02-04,2022-07-14,76
Belarus,2022-02-17,2022-07-14,22
Belgium,2021-02-04,2022-07-14,76
...,...,...,...
United Kingdom,2021-02-04,2022-07-14,76
United States,2021-02-04,2022-07-14,76
Uruguay,2021-02-04,2022-07-14,76


### Kiểm tra dữ liệu trước khi bỏ dòng nghệ sĩ lặp
Các dòng cùng vị trí chart phải có cùng bài, lượt nghe và cờ bài mới. Sai lệch âm thanh dưới 1e-9 được chấp nhận vì có thể do cách lưu số thực.

In [23]:
slot_check_cols = ['track_id', 'streams', 'is_new_entry']
df_slot_consistency = df_charts_clean.groupby(slot_grain_cols, observed=True)[slot_check_cols].nunique(dropna=False)
assert not df_slot_consistency.gt(1).any().any(), 'Slot có track hoặc streams/trạng thái xung đột.'
slot_audio_groups = df_charts_clean.groupby(slot_grain_cols, observed=True)[audio_cols]
df_slot_audio_spread = slot_audio_groups.max() - slot_audio_groups.min()
display(df_slot_audio_spread.max().to_frame('max_audio_difference'))
assert df_slot_audio_spread.le(1e-9).all().all(), 'Audio slot xung đột vượt sai số biểu diễn.'
print(f'số vị trí đã kiểm tra ({len(df_slot_consistency):,}):')
del df_slot_consistency, df_slot_audio_spread, slot_audio_groups

,max_audio_difference
danceability,0.000000e+00
energy,1.110223e-16
key,0.000000e+00
mode,0.000000e+00
loudness,0.000000e+00
speechiness,1.110223e-16
acousticness,1.110223e-16
instrumentalness,0.000000e+00
liveness,1.110223e-16
valence,0.000000e+00


số vị trí đã kiểm tra (1,109,375):


### Tạo bảng vị trí xếp hạng không lặp
Giữ một dòng cho mỗi thị trường, tuần và thứ hạng trong bảng đã làm sạch. Tổng lượt nghe được kiểm tra bằng một cách tính riêng.

In [24]:
df_slots = df_charts_clean.drop_duplicates(slot_grain_cols).copy()
assert not df_slots.duplicated(slot_grain_cols).any()
assert not df_slots.duplicated(['track_id', 'chart_country', 'week']).any()
slot_stream_reference = df_charts_clean.groupby(slot_grain_cols, observed=True)['streams'].first().sum()
assert df_slots['streams'].sum() == slot_stream_reference
print(f'số dòng trước loại lặp ({len(df_charts_clean):,}) và sau loại lặp ({len(df_slots):,}):')
print(f'tổng lượt nghe sau loại lặp nghệ sĩ ({int(slot_stream_reference):,}):')

số dòng trước loại lặp (1,787,531) và sau loại lặp (1,109,375):
tổng lượt nghe sau loại lặp nghệ sĩ (756,816,077,032):


### Đối chiếu số nghệ sĩ của mỗi bài
So sánh số dòng nghệ sĩ thực tế với số nghệ sĩ được khai báo. Những trường hợp khác nhau được lưu lại để xem xét.

In [25]:
df_artist_counts = df_charts_clean.groupby(['track_id', 'chart_country', 'week'], observed=True).size().rename('observed_artists').reset_index()
df_artist_check = df_slots[['track_id', 'chart_country', 'week', 'artists_num']].merge(
    df_artist_counts, on=['track_id', 'chart_country', 'week'], validate='one_to_one')
assert len(df_artist_check) == len(df_slots)
artist_exception_mask = df_artist_check['artists_num'].ne(df_artist_check['observed_artists'])
df_artist_exceptions = df_artist_check.loc[artist_exception_mask].copy()
assert len(df_artist_exceptions) == int(artist_exception_mask.sum())
print(f'số nhóm ngoại lệ ({len(df_artist_exceptions):,}) trên {len(df_artist_check):,} nhóm:')
display(df_artist_exceptions)

số nhóm ngoại lệ (2) trên 1,109,375 nhóm:


,track_id,chart_country,week,artists_num,observed_artists
751634,0GEeVI8Wa1KXGTStgBUerG,Norway,2022-05-19,2,1
752282,0GEeVI8Wa1KXGTStgBUerG,Norway,2021-05-20,2,1


### Tạo bảng thông tin bài hát
Kiểm tra ngày phát hành và âm thanh có thống nhất theo mã bài hay không. Sau đó giữ một dòng mỗi bài để tạo bảng thông tin bài hát.

In [26]:
track_audio_groups = df_charts_clean.groupby('track_id', observed=True)[audio_cols]
df_track_audio_spread = track_audio_groups.max() - track_audio_groups.min()
assert df_track_audio_spread.le(1e-9).all().all(), 'Audio cùng track xung đột.'
release_distinct = df_charts_clean.groupby('track_id', observed=True)['release_date'].nunique(dropna=False)
assert release_distinct.le(1).all(), 'Ngày phát hành cùng track xung đột.'
track_dim_cols = ['track_id', 'track_name', 'artist_id', 'artist_names', 'artists_num', 'artist_genre',
                  'release_date', 'release_date_clean', 'release_year'] + audio_cols
track_extra_cols = ['release_date_precision', 'genre_missing', 'audio_review_flag']
df_tracks = df_charts_clean[track_dim_cols + track_extra_cols].drop_duplicates('track_id').copy()
df_tracks['artist_metadata_policy'] = 'first_observed_artist; see track_artist_bridge'
assert df_tracks['track_id'].is_unique
print(f'dòng charts ({len(df_charts_clean):,}) và bài sau loại lặp ({len(df_tracks):,}):')
display(df_tracks.head(3))
del df_track_audio_spread, track_audio_groups

dòng charts (1,787,531) và bài sau loại lặp (39,190):


,track_id,track_name,artist_id,artist_names,artists_num,artist_genre,release_date,release_date_clean,release_year,danceability,energy,key,mode,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration,release_date_precision,genre_missing,audio_review_flag,artist_metadata_policy
0,2gpQi3hbcUAcEG8m2dlgfB,Plan A,3vQ0GE3mI0dAaxIMYe5g7z,Paulo Londra,1,argentine hip hop,2022-03-23,2022-03-23,2022,0.583,0.834,0,1,-4.875,0.0444,0.0495,0.0,0.0658,0.557,173.935,178203.0,day,False,False,first_observed_artist; see track_artist_bridge
1,2x8oBuYaObjqHqgGuIUZ0b,ARRANCARMELO,5YCc6xS5Gpj3EkaYGdjyNK,WOS,1,argentine indie,2022-04-06,2022-04-06,2022,0.654,0.354,5,1,-7.358,0.0738,0.7240,0.0,0.1340,0.262,81.956,183547.0,day,False,False,first_observed_artist; see track_artist_bridge
2,2SJZdZ5DLtlRosJ2xHJJJa,Chance,3vQ0GE3mI0dAaxIMYe5g7z,Paulo Londra,1,argentine hip hop,2022-04-06,2022-04-06,2022,0.721,0.463,1,0,-9.483,0.0646,0.2410,0.0,0.0929,0.216,137.915,204003.0,day,False,False,first_observed_artist; see track_artist_bridge


### Tạo bảng liên kết bài hát và nghệ sĩ
Giữ các cặp mã bài và mã nghệ sĩ duy nhất. Bảng này giúp biết những nghệ sĩ được ghi nhận cho từng bài mà không nhân thêm lượt nghe.

In [27]:
df_track_artist_bridge = df_charts_clean[['track_id', 'artist_id', 'artist_individual', 'artist_genre']].drop_duplicates(
    ['track_id', 'artist_id']).copy()
assert not df_track_artist_bridge.duplicated(['track_id', 'artist_id']).any()
print(f'dòng charts ({len(df_charts_clean):,}) và cặp bài nghệ sĩ ({len(df_track_artist_bridge):,}):')
display(df_track_artist_bridge.head(3))

dòng charts (1,787,531) và cặp bài nghệ sĩ (62,303):


,track_id,artist_id,artist_individual,artist_genre
0,2gpQi3hbcUAcEG8m2dlgfB,3vQ0GE3mI0dAaxIMYe5g7z,Paulo Londra,argentine hip hop
1,2x8oBuYaObjqHqgGuIUZ0b,5YCc6xS5Gpj3EkaYGdjyNK,WOS,argentine indie
2,2SJZdZ5DLtlRosJ2xHJJJa,3vQ0GE3mI0dAaxIMYe5g7z,Paulo Londra,argentine hip hop


### Liệt kê bài bị loại vì lỗi âm thanh
Đối chiếu danh sách bài trong lịch sử chart với bảng bài hát đã làm sạch. Các bài bị mất được lưu để giải thích, còn thứ hạng của chúng vẫn được dùng khi kiểm tra nhãn.

In [28]:
df_history_tracks = df_slots_observed[['track_id', 'track_name']].drop_duplicates('track_id').copy()
assert df_history_tracks['track_id'].is_unique
df_audio_lost_tracks = df_history_tracks.loc[~df_history_tracks['track_id'].isin(df_tracks['track_id'])].copy()
assert len(df_audio_lost_tracks) + int(df_history_tracks['track_id'].isin(df_tracks['track_id']).sum()) == len(df_history_tracks)
print(f'bài trước và sau dimension audio ({len(df_history_tracks):,} → {len(df_tracks):,}):')
display(df_audio_lost_tracks)

bài trước và sau dimension audio (39,202 → 39,190):


,track_id,track_name
6910,5saUoeceT2zau6ScmU77e6,Que a Pasao - Remix
92978,59o6ojGNGJOYiVJSzC6Lsa,Memories (feat. Kid Cudi) - 2021 Remix
244919,6zQhJcyuZGX7ADNMZF1VHL,14 De Febrero
531402,1HEKgcErkl1Ay9H4vMZHDJ,Balafre
808241,0ENwKWCMQmWb9b4hSqSVNW,Ef ástin er hrein
848348,1azXDEz0VyJzPUfjq3dxJZ,Rela
908733,02QztbIMzjvPNQnoECkydU,DOMENICA
910927,4WJVLEkcMMV2tHo1Bd65VN,DOMENICA
1102709,1ThaPy4W188i5SFdjXl64J,Casablanca
1213794,5KwFoU4ZlBbuKvNosAjYKo,Maria Maria


### Lấy lượt nghe Global theo bài và tuần
Lấy dữ liệu Global từ bảng đã bỏ dòng nghệ sĩ lặp. Mỗi bài chỉ có một dòng Global trong một tuần.

In [29]:
global_slot_mask = df_slots['chart_country'].eq('Global')
df_global_streams = df_slots.loc[global_slot_mask, ['track_id', 'week', 'streams']].copy()
assert len(df_global_streams) == int(global_slot_mask.sum())
df_global_streams['global_streams'] = df_global_streams['streams']
df_global_streams = df_global_streams[['track_id', 'week', 'global_streams']]
assert not df_global_streams.duplicated(track_week_cols).any()
print(f'số vị trí trước lọc ({len(df_slots):,}) và global sau lọc ({len(df_global_streams):,}):')
display(df_global_streams.head(3))

số vị trí trước lọc (1,109,375) và global sau lọc (57,993):


,track_id,week,global_streams
573695,6MWtB6iiXyIwun0YzU6DFP,2019-01-17,29944242
573696,2rPE9A1vEgShuZxxzR2tZH,2019-01-17,27807232
573697,5p7ujcrUXASCNwRaWNHR1C,2019-01-17,27138512


### Cộng lượt nghe quốc gia theo bài và tuần
Cộng lượt nghe của các quốc gia trong cùng bài và tuần, không bao gồm Global. Đây chỉ là tổng của các quốc gia có trong dữ liệu.

In [30]:
country_slot_mask = df_slots['chart_country'].ne('Global')
df_slots_country = df_slots.loc[country_slot_mask].copy()
assert len(df_slots_country) == int(country_slot_mask.sum())
df_country_streams = df_slots_country.groupby(track_week_cols, observed=True).agg(
    country_streams=('streams', 'sum'), country_count=('chart_country', 'nunique')).reset_index()
assert not df_country_streams.duplicated(track_week_cols).any()
assert df_country_streams['country_streams'].sum() == df_slots_country['streams'].sum()
print(f'số vị trí trước lọc ({len(df_slots):,}), quốc gia ({len(df_slots_country):,}), bài tuần ({len(df_country_streams):,}):')
display(df_country_streams.head(3))

số vị trí trước lọc (1,109,375), quốc gia (1,051,382), bài tuần (415,820):


,track_id,week,country_streams,country_count
0,000RW47rhEkSqjgTrZx7YX,2021-04-29,3435653,1
1,000RW47rhEkSqjgTrZx7YX,2021-05-06,3180293,1
2,000RW47rhEkSqjgTrZx7YX,2021-05-13,2984333,1


### Ghép lượt nghe Global với quốc gia
Ghép hai bảng theo mã bài và tuần để giữ cả bài chỉ có Global hoặc chỉ có quốc gia. Khóa duy nhất được kiểm tra để tránh nhân dòng.

In [31]:
df_track_week_performance = df_global_streams.merge(
    df_country_streams, on=track_week_cols, how='outer', validate='one_to_one', indicator=True)
overlap_week_count = int(df_track_week_performance['_merge'].eq('both').sum())
assert len(df_track_week_performance) == len(df_global_streams) + len(df_country_streams) - overlap_week_count
assert not df_track_week_performance.duplicated(track_week_cols).any()
print(f'global ({len(df_global_streams):,}), quốc gia ({len(df_country_streams):,}), sau ghép ({len(df_track_week_performance):,}):')
display(df_track_week_performance['_merge'].value_counts().to_frame('track_week_count'))

global (57,993), quốc gia (415,820), sau ghép (458,621):


,track_week_count
_merge,
right_only,400628
left_only,42801
both,15192


### Chọn nguồn lượt nghe để tránh cộng hai lần
Nếu bài có dữ liệu Global trong tuần thì dùng Global; nếu không mới dùng tổng quốc gia. Không cộng hai nguồn cho cùng một bài và tuần.

In [32]:
global_observed = df_track_week_performance['global_streams'].notna()
# Dùng Global khi có dữ liệu, tránh cộng lại lượt nghe quốc gia.
df_track_week_performance['streams_selected'] = df_track_week_performance['global_streams'].combine_first(
    df_track_week_performance['country_streams']).astype('int64')
df_track_week_performance['stream_source'] = np.where(global_observed, 'global', 'country')
df_track_week_performance['is_global_week'] = global_observed
assert df_track_week_performance['streams_selected'].notna().all()
assert df_track_week_performance.loc[global_observed, 'streams_selected'].eq(
    df_track_week_performance.loc[global_observed, 'global_streams']).all()
assert df_track_week_performance['track_id'].nunique() == df_slots['track_id'].nunique()
display(df_track_week_performance.head(3))

,track_id,week,global_streams,country_streams,country_count,_merge,streams_selected,stream_source,is_global_week
0,000RW47rhEkSqjgTrZx7YX,2021-04-29,NaN,3435653.0,1.0,right_only,3435653,country,False
1,000RW47rhEkSqjgTrZx7YX,2021-05-06,NaN,3180293.0,1.0,right_only,3180293,country,False
2,000RW47rhEkSqjgTrZx7YX,2021-05-13,NaN,2984333.0,1.0,right_only,2984333,country,False


### Đối chiếu tổng lượt nghe sau khi bỏ phần trùng
Kiểm tra tổng mới bằng tổng ban đầu trừ phần quốc gia đã chồng lên Global. Các bảng phân tích từng thị trường vẫn giữ lượt nghe của thị trường đó.

In [33]:
total_streams_original = int(df_slots['streams'].sum())
overlap_country_streams = int(df_track_week_performance.loc[global_observed, 'country_streams'].fillna(0).sum())
total_streams_selected = int(df_track_week_performance['streams_selected'].sum())
assert total_streams_selected == total_streams_original - overlap_country_streams
df_streams_audit = pd.DataFrame({
    'metric': ['original_slot_total', 'overlap_country_removed', 'selected_total', 'track_week_rows'],
    'value': [total_streams_original, overlap_country_streams, total_streams_selected, len(df_track_week_performance)]
})
print(f'lượt nghe cũ ({total_streams_original:,}), loại chồng lặp ({overlap_country_streams:,}), mới ({total_streams_selected:,}):')
display(df_streams_audit)

lượt nghe cũ (756,816,077,032), loại chồng lặp (105,653,132,647), mới (651,162,944,385):


,metric,value
0,original_slot_total,756816077032
1,overlap_country_removed,105653132647
2,selected_total,651162944385
3,track_week_rows,458621


### Tổng hợp thành tích bài hát để phân tích
Tính tổng lượt nghe đã xử lý chồng lấn và số tuần xuất hiện của từng bài. Các chỉ số toàn bộ lịch sử này chỉ dùng để mô tả, không làm đầu vào dự báo tuần đầu.

In [34]:
df_track_performance = df_track_week_performance.groupby('track_id', observed=True).agg(
    total_streams=('streams_selected', 'sum'), nweek=('week', 'nunique'),
    global_week_count=('is_global_week', 'sum'), first_chart_week=('week', 'min'), last_chart_week=('week', 'max')).reset_index()
df_track_performance['country_week_count'] = df_track_performance['nweek'] - df_track_performance['global_week_count']
df_track_performance['global_week_share'] = df_track_performance['global_week_count'] / df_track_performance['nweek']
df_track_performance['stream_scope'] = np.select(
    [df_track_performance['global_week_count'].eq(0), df_track_performance['country_week_count'].eq(0)],
    ['country_only', 'global_only'], default='mixed')
assert df_track_performance['track_id'].is_unique
assert df_track_performance['total_streams'].sum() == total_streams_selected
display(df_track_performance['stream_scope'].value_counts().to_frame('track_count'))

,track_count
stream_scope,
country_only,33843
global_only,3513
mixed,1834


### Bổ sung số quốc gia và thứ hạng tốt nhất
Ghép số quốc gia có bài và thứ hạng tốt nhất quan sát được vào bảng thành tích. Những chỉ số này cũng chỉ phục vụ phân tích lịch sử.

In [35]:
df_country_counts = df_slots_country.groupby('track_id', observed=True)['chart_country'].nunique().rename('n_countries').reset_index()
df_peak_ranks = df_slots.groupby('track_id', observed=True)['rank'].min().rename('peak_rank').reset_index()
performance_before = len(df_track_performance)
df_track_performance = df_track_performance.merge(df_country_counts, on='track_id', how='left', validate='one_to_one')
assert len(df_track_performance) == performance_before
df_track_performance['n_countries'] = df_track_performance['n_countries'].fillna(0).astype('int64')
df_track_performance = df_track_performance.merge(df_peak_ranks, on='track_id', how='left', validate='one_to_one')
assert len(df_track_performance) == performance_before
df_track_performance['log_streams'] = np.log1p(df_track_performance['total_streams'])
print(f'số bài trước và sau ghép ({performance_before:,} → {len(df_track_performance):,}):')
display(df_track_performance.head(3))

số bài trước và sau ghép (39,190 → 39,190):


,track_id,total_streams,nweek,global_week_count,first_chart_week,last_chart_week,country_week_count,global_week_share,stream_scope,n_countries,peak_rank,log_streams
0,000RW47rhEkSqjgTrZx7YX,37158272,26,0,2021-04-29,2021-10-21,26,0.0,country_only,1,15,17.430697
1,000xQL6tZNLJzIrtIgxqSl,107527761,17,17,2017-03-30,2017-07-20,0,1.0,global_only,0,46,18.493260
2,001b8t3bYPfnabpjpfG1Y4,160785,1,0,2021-02-04,2021-02-04,1,0.0,country_only,1,176,11.987830


### Giải phóng các bảng trung gian
Xóa các bản sao đã hoàn thành kiểm tra để giảm bộ nhớ. Các bảng sạch và bảng kiểm tra cần dùng về sau vẫn được giữ.

In [36]:
import gc
del df_charts_global, df_charts_country, df_slots_country
gc.collect()
print(f'đã giải phóng các bản sao tách thị trường:')

đã giải phóng các bản sao tách thị trường:


## Phần 4: Xử lý mã quốc gia và GDP
Nạp từng bảng, kiểm tra cấu trúc rồi tạo bảng liên kết tên thị trường với mã quốc gia. Giữ những trường hợp bổ sung thủ công và đơn vị chưa được xác minh.

### Nạp bảng mã quốc gia
Đọc file mã quốc gia của World Bank để dùng khi ghép GDP. Xem những dòng đầu để nắm các cột vùng và nhóm thu nhập.

In [37]:
df_country_codes_raw = pd.read_csv(path_raw / 'country_codes.csv')
print(f'bảng mã quốc gia ({len(df_country_codes_raw):,} dòng):')
display(df_country_codes_raw.head(3))
display(df_country_codes_raw.tail(3))

bảng mã quốc gia (217 dòng):


,country_code,region,income_group
0,ABW,Latin America & Caribbean,High income
1,AFG,South Asia,Low income
2,AGO,Sub-Saharan Africa,Lower middle income


,country_code,region,income_group
214,ZAF,Sub-Saharan Africa,Upper middle income
215,ZMB,Sub-Saharan Africa,Lower middle income
216,ZWE,Sub-Saharan Africa,Lower middle income


### Kiểm tra và chuẩn hóa bảng mã quốc gia
Kiểm tra mã quốc gia không bị lặp và tạo các cột có tiền tố wb. Các cột gốc được giữ để tiện đối chiếu với file nguồn.

In [38]:
assert not df_country_codes_raw['country_code'].duplicated().any()
df_country_codes_clean = df_country_codes_raw.copy()
df_country_codes_clean['wb_region'] = df_country_codes_clean['region']
df_country_codes_clean['wb_income_group'] = df_country_codes_clean['income_group']
print(f'phân bố vùng và nhóm thu nhập:')
display(df_country_codes_clean['wb_region'].value_counts(dropna=False).to_frame('country_count'))
display(df_country_codes_clean['wb_income_group'].value_counts(dropna=False).to_frame('country_count'))

phân bố vùng và nhóm thu nhập:


,country_count
wb_region,
Europe & Central Asia,58
Sub-Saharan Africa,48
Latin America & Caribbean,42
East Asia & Pacific,37
Middle East & North Africa,21
South Asia,8
North America,3


,country_count
wb_income_group,
High income,82
Upper middle income,55
Lower middle income,54
Low income,26


### Nạp bảng GDP
Đọc GDP theo quốc gia và năm, rồi xem những dòng đầu và cuối. Giá trị được giữ theo nguồn vì đơn vị của snapshot chưa được xác minh.

In [39]:
df_gdp_raw = pd.read_csv(path_raw / 'gdp_data.csv')
print(f'bảng gdp ({len(df_gdp_raw):,} dòng):')
display(df_gdp_raw.head(3))
display(df_gdp_raw.tail(3))

bảng gdp (13,365 dòng):


,country_name,country_code,year,value
0,Afghanistan,AFG,1960,537777811.1
1,Afghanistan,AFG,1961,548888895.6
2,Afghanistan,AFG,1962,546666677.8


,country_name,country_code,year,value
13362,Zimbabwe,ZWE,2020,2.150970e+10
13363,Zimbabwe,ZWE,2021,2.837124e+10
13364,Zimbabwe,ZWE,2022,2.067806e+10


### Kiểm tra khóa và kiểu dữ liệu GDP
Chuyển năm và giá trị GDP sang kiểu số, kiểm tra cặp mã quốc gia và năm không lặp. Giá trị GDP phải có số liệu và không âm.

In [40]:
df_gdp_clean = df_gdp_raw.copy()
df_gdp_clean['year'] = pd.to_numeric(df_gdp_clean['year'], errors='raise').astype('int16')
df_gdp_clean['value'] = pd.to_numeric(df_gdp_clean['value'], errors='raise')
assert not df_gdp_clean.duplicated(['country_code', 'year']).any()
assert df_gdp_clean['value'].notna().all() and df_gdp_clean['value'].ge(0).all()
print(f'thống kê năm và giá trị gdp:')
display(df_gdp_clean[['year', 'value']].describe().T)

thống kê năm và giá trị gdp:


,count,mean,std,min,25%,50%,75%,max
year,13365.0,1.994479e+03,1.745958e+01,1960.000,1.980000e+03,1.996000e+03,2.009000e+03,2.022000e+03
value,13365.0,1.207051e+12,5.413230e+12,8824746.238,2.523472e+09,1.842778e+10,2.240000e+11,1.010000e+14


### Ghép tên quốc gia với thông tin World Bank
Lấy tên và mã quốc gia từ bảng GDP rồi ghép vùng và nhóm thu nhập. Thông tin này được dùng để nối các thị trường Spotify với dữ liệu quốc gia.

In [41]:
df_wb_names = df_gdp_clean[['country_name', 'country_code']].drop_duplicates()
assert df_wb_names['country_name'].is_unique and df_wb_names['country_code'].is_unique
print(f'cặp tên mã trước và sau bỏ trùng ({len(df_gdp_clean):,} → {len(df_wb_names):,}):')
df_wb_names = df_wb_names.merge(df_country_codes_clean[['country_code', 'wb_region', 'wb_income_group']],
                               on='country_code', how='left', validate='one_to_one')
assert len(df_wb_names) == df_gdp_clean['country_code'].nunique()
print(f'tên world bank sau ghép ({len(df_wb_names):,}):')
display(df_wb_names.head(3))

cặp tên mã trước và sau bỏ trùng (13,365 → 262):
tên world bank sau ghép (262):


,country_name,country_code,wb_region,wb_income_group
0,Afghanistan,AFG,South Asia,Low income
1,Africa Eastern and Southern,AFE,NaN,NaN
2,Africa Western and Central,AFW,NaN,NaN


### Tạo bảng liên kết thị trường với mã quốc gia
Chuẩn hóa tên thị trường để ghép với World Bank. Thông tin Đài Loan được bổ sung theo ánh xạ của dự án và ghi rõ nguồn bổ sung.

In [42]:
df_country_lookup = df_slots.loc[df_slots['chart_country'].ne('Global'), ['chart_country']].drop_duplicates()
assert len(df_country_lookup) == df_slots.loc[df_slots['chart_country'].ne('Global'), 'chart_country'].nunique()
df_country_lookup['country_name'] = df_country_lookup['chart_country'].astype('str').replace(COUNTRY_ALIAS_MAP)
lookup_rows = len(df_country_lookup)
df_country_lookup = df_country_lookup.merge(df_wb_names, on='country_name', how='left', validate='one_to_one')
assert len(df_country_lookup) == lookup_rows
df_country_lookup['metadata_source'] = 'World Bank metadata snapshot'
taiwan_mask = df_country_lookup['chart_country'].eq('Taiwan')
df_country_lookup.loc[taiwan_mask, ['country_code', 'wb_region', 'wb_income_group']] = ['TWN', 'East Asia & Pacific', 'High income']
df_country_lookup.loc[taiwan_mask, 'metadata_source'] = 'manual project mapping'
assert df_country_lookup[['country_code', 'wb_region', 'wb_income_group']].notna().all().all()
print(f'thị trường trước và sau ghép ({lookup_rows:,} → {len(df_country_lookup):,}):')
display(df_country_lookup)

thị trường trước và sau ghép (73 → 73):


,chart_country,country_name,country_code,wb_region,wb_income_group,metadata_source
0,Argentina,Argentina,ARG,Latin America & Caribbean,Upper middle income,World Bank metadata snapshot
1,Australia,Australia,AUS,East Asia & Pacific,High income,World Bank metadata snapshot
2,Austria,Austria,AUT,Europe & Central Asia,High income,World Bank metadata snapshot
3,Belarus,Belarus,BLR,Europe & Central Asia,Upper middle income,World Bank metadata snapshot
4,Belgium,Belgium,BEL,Europe & Central Asia,High income,World Bank metadata snapshot
...,...,...,...,...,...,...
68,Ukraine,Ukraine,UKR,Europe & Central Asia,Lower middle income,World Bank metadata snapshot
69,Uruguay,Uruguay,URY,Latin America & Caribbean,High income,World Bank metadata snapshot
70,United States,United States,USA,North America,High income,World Bank metadata snapshot
71,Venezuela,"Venezuela, RB",VEN,Latin America & Caribbean,Upper middle income,World Bank metadata snapshot


### Bổ sung vùng, ngôn ngữ và số tuần của thị trường
Ghép thông tin chart vào bảng liên kết quốc gia để dùng trên dashboard. Nếu một thị trường có nhiều giá trị khác nhau, trường đó được để trống và đánh dấu cần xem lại.

In [43]:
df_market_metadata = df_slots_observed.groupby('chart_country', observed=True).agg(
    chart_region=('chart_region', lambda values: values.iloc[0] if values.nunique(dropna=False) == 1 else pd.NA),
    chart_language=('chart_language', lambda values: values.iloc[0] if values.nunique(dropna=False) == 1 else pd.NA),
    n_weeks_observed=('week', 'nunique')).reset_index()
assert df_market_metadata['chart_country'].is_unique
df_market_metadata['metadata_ambiguous'] = df_market_metadata[['chart_region', 'chart_language']].isna().any(axis=1)
lookup_rows = len(df_country_lookup)
df_country_lookup = df_country_lookup.merge(df_market_metadata, on='chart_country', how='left', validate='one_to_one')
assert len(df_country_lookup) == lookup_rows
print(f'quốc gia trước và sau bổ sung thông tin ({lookup_rows:,} → {len(df_country_lookup):,}):')

quốc gia trước và sau bổ sung thông tin (73 → 73):


## Phần 5: Xử lý dữ liệu COVID
Tách thông tin tỉnh, bang, chuẩn hóa ngày và gộp số ca tử vong theo tuần kết thúc thứ Năm. Giữ cờ thiếu ngày và các lần điều chỉnh số liệu.

### Nạp và kiểm tra dữ liệu COVID hằng ngày
Tách dòng thông tin tỉnh, bang khỏi dữ liệu ngày rồi kiểm tra ngày và số ca tử vong. Số âm do điều chỉnh báo cáo được giữ lại.

In [44]:
df_covid_raw = pd.read_csv(path_raw / 'global_deaths.csv')
assert df_covid_raw.iloc[0, 0] == 'Province/State'
df_covid_daily_values = df_covid_raw.iloc[1:, 1:].apply(pd.to_numeric, errors='raise')
assert len(df_covid_daily_values) == len(df_covid_raw) - 1
assert df_covid_daily_values.notna().all().all()
assert np.isfinite(df_covid_daily_values).all().all()
covid_dates = pd.to_datetime(df_covid_raw.iloc[1:, 0], format='%m/%d/%y', errors='raise')
assert covid_dates.is_unique and covid_dates.is_monotonic_increasing
assert covid_dates.diff().dropna().eq(pd.Timedelta(days=1)).all()
print(f'dòng covid trước và sau tách thông tin ({len(df_covid_raw):,} → {len(df_covid_daily_values):,}):')
print(f'thời gian covid ({covid_dates.min().date()} đến {covid_dates.max().date()}):')

dòng covid trước và sau tách thông tin (1,143 → 1,142):
thời gian covid (2020-01-23 đến 2023-03-09):


### Gộp dữ liệu COVID về từng quốc gia
Dùng thông tin tỉnh, bang để gộp đúng quốc gia và tách Hong Kong. Tổng Global được tính trước khi chọn các thị trường có trong Spotify.

In [45]:
import re
covid_entities = []
covid_aliases = {'US': 'United States', 'Korea, South': 'Korea', 'Taiwan*': 'Taiwan', 'Czechia': 'Czech Republic'}
for column in df_covid_daily_values.columns:
    base_country = re.sub(r'\.\d+$', '', column)
    province = df_covid_raw.loc[0, column]
    country = 'Hong Kong' if province == 'Hong Kong' else covid_aliases.get(base_country, base_country)
    covid_entities.append(country)
df_covid_daily = df_covid_daily_values.T.groupby(covid_entities, sort=True).sum().T
assert np.allclose(df_covid_daily.sum(axis=1), df_covid_daily_values.sum(axis=1))
df_covid_daily['Global'] = df_covid_daily.sum(axis=1)
df_covid_daily.index = pd.DatetimeIndex(covid_dates)
df_covid_long = df_covid_daily.rename_axis('date').reset_index().melt(id_vars='date', var_name='country', value_name='daily_deaths')
assert len(df_covid_long) == len(covid_dates) * df_covid_daily.shape[1]
print(f'ngày ({len(df_covid_daily):,}) và thực thể gồm global ({df_covid_daily.shape[1]:,}):')

ngày (1,142) và thực thể gồm global (203):


### Tổng hợp số ca tử vong theo tuần
Gộp các ngày vào tuần kết thúc thứ Năm để khớp với chart. Giữ số ngày có dữ liệu và đánh dấu tuần chưa đủ bảy ngày.

In [46]:
df_covid_long['week'] = df_covid_long['date'] + pd.to_timedelta((3 - df_covid_long['date'].dt.weekday) % 7, unit='D')
df_covid_long['negative_daily_adjustment'] = df_covid_long['daily_deaths'].lt(0)
df_covid_weekly_all = df_covid_long.groupby(['country', 'week'], as_index=False).agg(
    weekly_deaths=('daily_deaths', 'sum'), observed_days=('date', 'nunique'),
    has_negative_adjustment=('negative_daily_adjustment', 'any'))
df_covid_weekly_all['partial_week'] = df_covid_weekly_all['observed_days'].lt(7)
df_covid_weekly_all['cumulative_deaths'] = df_covid_weekly_all.groupby('country')['weekly_deaths'].cumsum()
assert not df_covid_weekly_all.duplicated(['country', 'week']).any()
assert np.isclose(df_covid_weekly_all['weekly_deaths'].sum(), df_covid_long['daily_deaths'].sum())
display(df_covid_weekly_all['observed_days'].value_counts().to_frame('country_weeks'))

,country_weeks
observed_days,
7,33089
1,203


### Chọn các thị trường COVID cần ghép
Giữ dữ liệu của các quốc gia có trong chart và Global. Cờ điều chỉnh âm được giữ để phân biệt số liệu nguồn với lỗi xử lý.

In [47]:
covid_markets = set(df_country_lookup['chart_country']) | {'Global'}
covid_market_mask = df_covid_weekly_all['country'].isin(covid_markets)
df_covid_weekly_clean = df_covid_weekly_all.loc[covid_market_mask].copy()
assert len(df_covid_weekly_clean) == int(covid_market_mask.sum())
assert set(df_covid_weekly_clean['country']) == covid_markets
assert not df_covid_weekly_clean.duplicated(['country', 'week']).any()
print(f'covid trước và sau chọn thị trường ({len(df_covid_weekly_all):,} → {len(df_covid_weekly_clean):,}):')
display(df_covid_weekly_clean.loc[df_covid_weekly_clean['weekly_deaths'].lt(0)].head())

covid trước và sau chọn thị trường (33,292 → 12,136):


,country,week,weekly_deaths,observed_days,has_negative_adjustment,partial_week,cumulative_deaths
1495,Australia,2020-06-04,-1.0,7,True,False,102.0
1778,Austria,2022-09-15,-17.0,7,True,False,20661.0
1784,Austria,2022-10-27,-25.0,7,True,False,20946.0
2819,Belgium,2020-08-27,-92.0,7,True,False,9884.0
5524,Canada,2022-03-17,-137.0,7,True,False,37074.0


## Phần 6: Xử lý dữ liệu hạnh phúc
Chuẩn hóa tên quốc gia, năm và các cột thay đổi giữa những phiên bản báo cáo. Các khóa trùng chưa giải quyết được giữ riêng và không dùng khi ghép.

### Nạp và kiểm tra dữ liệu hạnh phúc
Đọc dữ liệu hạnh phúc, chuẩn hóa tên quốc gia, năm và điểm số. Các dòng trùng quốc gia và năm được lưu riêng để kiểm tra.

In [48]:
df_whs_raw = pd.read_csv(path_raw / 'whs_years_updated.csv')
df_whs = pd.DataFrame({'country': df_whs_raw['Country'].str.strip(),
                       'year': pd.to_numeric(df_whs_raw['Year'], errors='raise').astype('int16'),
                       'happiness_score': pd.to_numeric(df_whs_raw['Score'], errors='raise')})
whr_aliases = {'Czechia': 'Czech Republic', 'South Korea': 'Korea', 'Türkiye': 'Turkey',
               'Taiwan Province of China': 'Taiwan', 'Hong Kong S.A.R. of China': 'Hong Kong',
               'Slovak Republic': 'Slovakia', 'United States of America': 'United States',
               'Korea, Republic of': 'Korea', 'Taiwan, Province of China': 'Taiwan',
               'Bolivia, Plurinational State of': 'Bolivia', 'Viet Nam': 'Vietnam',
               'Venezuela, Bolivarian Republic of': 'Venezuela'}
df_whs['country'] = df_whs['country'].replace(whr_aliases)
assert df_whs[['country', 'year', 'happiness_score']].notna().all().all()
assert df_whs['happiness_score'].between(0, 10).all()
duplicate_whs_mask = df_whs.duplicated(['country', 'year'], keep=False)
df_whs_rejected = df_whs_raw.loc[duplicate_whs_mask].copy()
assert len(df_whs_rejected) == int(duplicate_whs_mask.sum())
print(f'dòng hạnh phúc ({len(df_whs):,}) và dòng khóa trùng ({len(df_whs_rejected):,}):')
display(df_whs_rejected)

dòng hạnh phúc (1,510) và dòng khóa trùng (2):


,Country,Region,Happiness Rank,Score,Standard Error,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption),Generosity,Dystopia Residual,Year,Lower Confidence Interval,Upper Confidence Interval,Happiness.Rank,Whisker.high,Whisker.low,Economy..GDP.per.Capita.,Health..Life.Expectancy.,Trust..Government.Corruption.,Dystopia.Residual,Overall rank,GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Perceptions of corruption,Regional indicator,Standard error of ladder score,upperwhisker,lowerwhisker,Logged GDP per capita,Ladder score in Dystopia,Explained by: Log GDP per capita,Explained by: Social support,Explained by: Healthy life expectancy,Explained by: Freedom to make life choices,Explained by: Generosity,Explained by: Perceptions of corruption,Dystopia + residual,RANK,Whisker-high,Whisker-low,Dystopia (1.83) + residual,Explained by: GDP per capita
1124,Cyprus,Western Europe,NaN,6.2209,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.909315,0.819165,0.448146,0.123327,0.061780,NaN,41.0,6.320243,6.121557,2.044133,1.815065
1162,Cyprus,Western Europe,NaN,5.4666,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.888054,0.819165,0.522693,0.130266,0.213033,NaN,79.0,5.608694,5.324506,1.078338,1.815065


### Hợp nhất các cột thứ hạng hạnh phúc
Gộp các tên cột thứ hạng của từng phiên bản báo cáo vào một cột chung. Nếu nguồn thiếu thứ hạng thì tính từ điểm số và ghi lại cách tính.

In [49]:
rank_sources = ['Happiness Rank', 'Happiness.Rank', 'Overall rank', 'RANK']
df_whs['happiness_rank'] = np.nan
df_whs['happiness_rank_source'] = pd.Series(pd.NA, index=df_whs.index, dtype='str')
for column in rank_sources:
    values = pd.to_numeric(df_whs_raw[column], errors='coerce')
    take_mask = df_whs['happiness_rank'].isna() & values.notna()
    df_whs.loc[take_mask, 'happiness_rank'] = values.loc[take_mask]
    df_whs.loc[take_mask, 'happiness_rank_source'] = column
rank_missing_mask = df_whs['happiness_rank'].isna()
derived_ranks = df_whs.groupby('year')['happiness_score'].rank(ascending=False, method='min')
df_whs.loc[rank_missing_mask, 'happiness_rank'] = derived_ranks.loc[rank_missing_mask]
df_whs.loc[rank_missing_mask, 'happiness_rank_source'] = 'derived from score'
assert df_whs['happiness_rank'].notna().all()
display(df_whs.groupby('year')['happiness_rank_source'].value_counts().to_frame('rows'))

,,rows
year,happiness_rank_source,
2015,Happiness Rank,158
2016,Happiness Rank,157
2017,Happiness.Rank,155
2018,Overall rank,156
2019,Overall rank,156
2020,derived from score,153
2021,derived from score,149
2022,RANK,146
2023,derived from score,137


### Khai báo các cột thành phần hạnh phúc
Liệt kê những tên cột tương ứng với sáu thành phần trong các năm báo cáo. Giữ thông tin cột gốc vì cách đo giữa các năm có thể khác nhau.

In [50]:
whr_factor_sources = {
    'happiness_gdp': ['Explained by: Log GDP per capita', 'Explained by: GDP per capita',
                      'GDP per capita', 'Economy (GDP per Capita)', 'Economy..GDP.per.Capita.'],
    'happiness_social': ['Explained by: Social support', 'Social support', 'Family'],
    'happiness_health': ['Explained by: Healthy life expectancy', 'Healthy life expectancy',
                         'Health (Life Expectancy)', 'Health..Life.Expectancy.'],
    'happiness_freedom': ['Explained by: Freedom to make life choices', 'Freedom to make life choices', 'Freedom'],
    'happiness_generosity': ['Explained by: Generosity', 'Generosity'],
    'happiness_corruption': ['Explained by: Perceptions of corruption', 'Perceptions of corruption',
                             'Trust (Government Corruption)', 'Trust..Government.Corruption.']}
display(pd.Series(whr_factor_sources).to_frame('source_columns'))

,source_columns
happiness_gdp,"[Explained by: Log GDP per capita, Explained b..."
happiness_social,"[Explained by: Social support, Social support,..."
happiness_health,"[Explained by: Healthy life expectancy, Health..."
happiness_freedom,"[Explained by: Freedom to make life choices, F..."
happiness_generosity,"[Explained by: Generosity, Generosity]"
happiness_corruption,"[Explained by: Perceptions of corruption, Perc..."


### Hợp nhất thành phần và tách khóa hạnh phúc bị trùng
Gộp các thành phần vào bảng sạch và giữ tên cột nguồn. Các khóa trùng chưa giải quyết được bị loại khỏi bảng ghép, nhưng vẫn lưu riêng để đối chiếu.

In [51]:
for target, candidates in whr_factor_sources.items():
    df_whs[target] = np.nan
    df_whs[target + '_source_column'] = pd.Series(pd.NA, index=df_whs.index, dtype='str')
    for column in candidates:
        values = pd.to_numeric(df_whs_raw[column], errors='coerce')
        take_mask = df_whs[target].isna() & values.notna()
        df_whs.loc[take_mask, target] = values.loc[take_mask]
        df_whs.loc[take_mask, target + '_source_column'] = column
df_whs_clean = df_whs.loc[~duplicate_whs_mask].copy()
assert len(df_whs_clean) + len(df_whs_rejected) == len(df_whs)
assert not df_whs_clean.duplicated(['country', 'year']).any()
print(f'hạnh phúc trước và sau cách ly ({len(df_whs):,} → {len(df_whs_clean):,}):')
display(df_whs_clean.isna().sum().to_frame('missing_count'))

hạnh phúc trước và sau cách ly (1,510 → 1,508):


,missing_count
country,0
year,0
happiness_score,0
happiness_rank,0
happiness_rank_source,0
happiness_gdp,3
happiness_gdp_source_column,3
happiness_social,3
happiness_social_source_column,3
happiness_health,4


### Quan sát điểm hạnh phúc theo năm
Thống kê số dòng và khoảng điểm của từng năm. Kết quả giúp nhìn rõ thời gian và thang điểm của nguồn trước khi ghép với chart.

In [52]:
df_happiness_by_year = df_whs.groupby('year', as_index=False).agg(
    row_count=('country', 'size'), min_score=('happiness_score', 'min'), max_score=('happiness_score', 'max'))
print(f'điểm hạnh phúc theo năm:')
display(df_happiness_by_year)

điểm hạnh phúc theo năm:


,year,row_count,min_score,max_score
0,2015,158,2.8390,7.5870
1,2016,157,2.9050,7.5260
2,2017,155,2.6930,7.5370
3,2018,156,2.9050,7.6320
4,2019,156,2.8530,7.7690
5,2020,153,2.5669,7.8087
6,2021,149,2.5230,7.8420
7,2022,146,2.4038,7.8210
8,2023,137,1.8590,7.8042
9,2024,143,1.7210,7.7407


## Phần 7: Ghép dữ liệu và tạo các bảng tổng hợp
Tạo bảng âm thanh theo tuần, quốc gia theo năm và nghệ sĩ theo thị trường. Các phép ghép đều kiểm tra số dòng và lưu trường hợp không khớp.

### Tạo hàm tổng hợp đặc trưng âm thanh
Tính trung bình thường và trung bình theo trọng số lượt nghe của các chỉ số âm thanh. Key được biểu diễn bằng tỷ lệ từng nhóm; nếu tổng lượt nghe bằng 0 thì dùng trung bình thường.

In [53]:
def aggregate_audio(df_input, group_cols):
    numeric_audio = [column for column in audio_cols if column != 'key']
    groups = df_input.groupby(group_cols, observed=True)
    df_result = groups[numeric_audio].mean()
    stream_totals = groups['streams'].sum()
    for column in numeric_audio:
        weighted_values = df_input[column].astype(float) * df_input['streams']
        weighted_sums = weighted_values.groupby([df_input[key] for key in group_cols], observed=True).sum()
        df_result[column + '_weighted'] = weighted_sums.div(stream_totals.where(stream_totals.gt(0))).fillna(df_result[column])
    for musical_key in range(-1, 12):
        key_indicator = df_input['key'].eq(musical_key).astype(float)
        df_result[f'key_{musical_key}_share'] = key_indicator.groupby([df_input[key] for key in group_cols], observed=True).mean()
    assert np.allclose(df_result[[f'key_{key}_share' for key in range(-1, 12)]].sum(axis=1), 1)
    return df_result.reset_index()
print(f'hàm tổng hợp đặc trưng theo khóa được chọn:')

hàm tổng hợp đặc trưng theo khóa được chọn:


### Tạo bảng xu hướng âm thanh theo tuần
Tổng hợp âm thanh và lượt nghe theo thị trường, tuần từ bảng không lặp nghệ sĩ. Kiểm tra số dòng sau ghép và tổng lượt nghe của bảng kết quả.

In [54]:
df_mart_weekly = df_slots.groupby(['chart_country', 'week'], observed=True, as_index=False).agg(
    total_streams=('streams', 'sum'), track_count=('track_id', 'nunique'),
    new_entries=('is_new_entry', 'sum'))
df_audio_weekly = aggregate_audio(df_slots, ['chart_country', 'week'])
weekly_rows = len(df_mart_weekly)
df_mart_weekly = df_mart_weekly.merge(df_audio_weekly, on=['chart_country', 'week'], validate='one_to_one')
assert len(df_mart_weekly) == weekly_rows
assert df_mart_weekly['total_streams'].sum() == df_slots['streams'].sum()
df_mart_weekly['year'] = df_mart_weekly['week'].dt.year
df_mart_weekly['month'] = df_mart_weekly['week'].dt.month
print(f'mart tuần trước và sau ghép âm thanh ({weekly_rows:,} → {len(df_mart_weekly):,}):')
display(df_mart_weekly.head(3))

mart tuần trước và sau ghép âm thanh (5,548 → 5,548):


,chart_country,week,total_streams,track_count,new_entries,danceability,energy,mode,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration,danceability_weighted,energy_weighted,mode_weighted,loudness_weighted,speechiness_weighted,acousticness_weighted,instrumentalness_weighted,liveness_weighted,valence_weighted,tempo_weighted,duration_weighted,key_-1_share,key_0_share,key_1_share,key_2_share,key_3_share,key_4_share,key_5_share,key_6_share,key_7_share,key_8_share,key_9_share,key_10_share,key_11_share,year,month
0,Argentina,2021-02-04,96192710,200,16,0.71143,0.657585,0.525,-5.661310,0.114690,0.232841,0.003564,0.167194,0.632594,126.869450,214014.76,0.735600,0.659894,0.496514,-5.377115,0.134070,0.232843,0.001755,0.151609,0.655807,129.344360,206121.042976,0.0,0.165,0.105,0.065,0.02,0.090,0.065,0.085,0.08,0.045,0.090,0.08,0.11,2021,2
1,Argentina,2021-02-11,98730688,200,13,0.71567,0.663280,0.550,-5.597865,0.115113,0.231033,0.003656,0.170700,0.642104,125.213295,216345.09,0.736972,0.661955,0.508641,-5.359143,0.132871,0.229692,0.001859,0.155172,0.660159,128.379880,207608.310346,0.0,0.165,0.105,0.070,0.02,0.085,0.055,0.085,0.09,0.045,0.090,0.09,0.10,2021,2
2,Argentina,2021-02-18,96438610,200,10,0.71548,0.668385,0.550,-5.543840,0.116869,0.225014,0.003658,0.172039,0.642410,125.374275,214453.14,0.735207,0.663016,0.515347,-5.340677,0.134443,0.228221,0.001860,0.161561,0.660596,127.782181,207663.596013,0.0,0.150,0.105,0.075,0.02,0.080,0.060,0.095,0.09,0.050,0.095,0.09,0.09,2021,2


### Ghép COVID vào bảng xu hướng tuần
Ghép dữ liệu theo thị trường và tuần, rồi kiểm tra những dòng không khớp. Tuần nằm trước thời gian nguồn COVID được giữ trống và ghi lý do.

In [55]:
df_mart_weekly = df_mart_weekly.merge(df_covid_weekly_clean, left_on=['chart_country', 'week'],
    right_on=['country', 'week'], how='left', validate='one_to_one', indicator='covid_join')
assert len(df_mart_weekly) == weekly_rows
df_mart_weekly['covid_observed'] = df_mart_weekly['covid_join'].eq('both')
df_mart_weekly['covid_missing_reason'] = np.where(df_mart_weekly['covid_observed'], 'matched',
    np.where(df_mart_weekly['week'].lt(df_covid_weekly_clean['week'].min()), 'before_source_period', 'unmatched_within_period'))
assert not df_mart_weekly['covid_missing_reason'].eq('unmatched_within_period').any()
df_covid_unmatched = df_mart_weekly.loc[~df_mart_weekly['covid_observed'], ['chart_country', 'week', 'covid_missing_reason']].copy()
assert len(df_covid_unmatched) == int((~df_mart_weekly['covid_observed']).sum())
print(f'mart tuần sau ghép ({len(df_mart_weekly):,}) và tuần thiếu covid ({len(df_covid_unmatched):,}):')
display(df_mart_weekly['covid_missing_reason'].value_counts().to_frame('rows'))

mart tuần sau ghép (5,548) và tuần thiếu covid (160):


,rows
covid_missing_reason,
matched,5388
before_source_period,160


### Tạo bảng âm thanh quốc gia theo năm
Tổng hợp các chỉ số âm thanh và lượt nghe theo quốc gia, năm. Giữ số tuần và số vị trí để thấy mức độ đầy đủ của từng năm.

In [56]:
country_slot_mask = df_slots['chart_country'].ne('Global')
df_country_slots = df_slots.loc[country_slot_mask, ['chart_country', 'year', 'track_id', 'week', 'streams'] + audio_cols].copy()
assert len(df_country_slots) == int(country_slot_mask.sum())
df_mart_country_year = df_country_slots.groupby(['chart_country', 'year'], observed=True, as_index=False).agg(
    total_streams=('streams', 'sum'), track_count=('track_id', 'nunique'),
    n_weeks=('week', 'nunique'), slot_count=('track_id', 'size'))
df_audio_country_year = aggregate_audio(df_country_slots, ['chart_country', 'year'])
annual_rows = len(df_mart_country_year)
df_mart_country_year = df_mart_country_year.merge(df_audio_country_year, on=['chart_country', 'year'], validate='one_to_one')
assert len(df_mart_country_year) == annual_rows
assert df_mart_country_year['total_streams'].sum() == df_country_slots['streams'].sum()
print(f'vị trí trước và sau chọn quốc gia ({len(df_slots):,} → {len(df_country_slots):,}):')
print(f'mart năm trước và sau ghép âm thanh ({annual_rows:,} → {len(df_mart_country_year):,}):')

vị trí trước và sau chọn quốc gia (1,109,375 → 1,051,382):
mart năm trước và sau ghép âm thanh (141 → 141):


### Ghép mã và thông tin quốc gia vào bảng năm
Ghép bảng liên kết quốc gia vào bảng tổng hợp năm. Mỗi quốc gia chỉ có một dòng thông tin nên phép ghép không được làm tăng số dòng.

In [57]:
df_mart_country_year = df_mart_country_year.merge(df_country_lookup, on='chart_country', how='left', validate='many_to_one')
assert len(df_mart_country_year) == annual_rows
assert df_mart_country_year['country_code'].notna().all()
print(f'mart năm trước và sau ghép mã ({annual_rows:,} → {len(df_mart_country_year):,}):')
display(df_mart_country_year[['chart_country', 'country_code', 'wb_region', 'metadata_source']].drop_duplicates().head())

mart năm trước và sau ghép mã (141 → 141):


,chart_country,country_code,wb_region,metadata_source
0,Argentina,ARG,Latin America & Caribbean,World Bank metadata snapshot
2,Australia,AUS,East Asia & Pacific,World Bank metadata snapshot
4,Austria,AUT,Europe & Central Asia,World Bank metadata snapshot
6,Belarus,BLR,Europe & Central Asia,World Bank metadata snapshot
7,Belgium,BEL,Europe & Central Asia,World Bank metadata snapshot


### Ghép GDP theo mã quốc gia và năm
Ghép GDP bằng mã quốc gia và năm, rồi lưu các trường hợp chưa khớp. Giá trị GDP được giữ theo file nguồn.

In [58]:
df_gdp_join = df_gdp_clean[['country_code', 'year', 'value']].copy()
df_gdp_join['gdp_source_value'] = df_gdp_join['value']
df_mart_country_year = df_mart_country_year.merge(df_gdp_join, on=['country_code', 'year'],
    how='left', validate='one_to_one', indicator='gdp_join')
assert len(df_mart_country_year) == annual_rows
df_gdp_unmatched = df_mart_country_year.loc[df_mart_country_year['gdp_join'].eq('left_only'), ['chart_country', 'country_code', 'year']].copy()
assert len(df_gdp_unmatched) == int(df_mart_country_year['gdp_join'].eq('left_only').sum())
print(f'mart năm sau ghép ({len(df_mart_country_year):,}) và gdp không khớp ({len(df_gdp_unmatched):,}):')
display(df_gdp_unmatched)

mart năm sau ghép (141) và gdp không khớp (3):


,chart_country,country_code,year
122,Taiwan,TWN,2021
123,Taiwan,TWN,2022
138,Venezuela,VEN,2022


### Ghép điểm hạnh phúc vào bảng quốc gia năm
Ghép theo quốc gia và năm báo cáo để giữ toàn bộ các dòng của bảng năm. Trường hợp thiếu hoặc có khóa trùng chưa giải quyết được vẫn để trống.

In [59]:
df_mart_country_year = df_mart_country_year.merge(df_whs_clean, left_on=['chart_country', 'year'],
    right_on=['country', 'year'], how='left', validate='one_to_one', indicator='whr_join')
assert len(df_mart_country_year) == annual_rows
df_whr_unmatched = df_mart_country_year.loc[df_mart_country_year['whr_join'].eq('left_only'), ['chart_country', 'year']].copy()
assert len(df_whr_unmatched) == int(df_mart_country_year['whr_join'].eq('left_only').sum())
print(f'mart năm sau ghép ({len(df_mart_country_year):,}) và hạnh phúc không khớp ({len(df_whr_unmatched):,}):')
display(df_whr_unmatched)

mart năm sau ghép (141) và hạnh phúc không khớp (1):


,chart_country,year
24,Cyprus,2022


### Tạo bảng thành tích nghệ sĩ theo thị trường
Tổng hợp số bài, số tuần, lượt nghe và thứ hạng tốt nhất của từng nghệ sĩ trong thị trường. Bài hợp tác được tính đủ lượt nghe cho từng nghệ sĩ, nên không cộng bảng này để lấy tổng lượt nghe bài hát.

In [60]:
df_mart_artist_chart = df_charts_clean.groupby(['artist_id', 'chart_country'], observed=True, as_index=False).agg(
    artist_individual=('artist_individual', 'first'), total_streams=('streams', 'sum'),
    track_count=('track_id', 'nunique'), n_weeks=('week', 'nunique'),
    peak_rank=('rank', 'min'), first_chart_week=('week', 'min'), last_chart_week=('week', 'max'))
df_mart_artist_chart['streams_credit_policy'] = 'full credit per collaborating artist; non-additive'
assert not df_mart_artist_chart.duplicated(['artist_id', 'chart_country']).any()
print(f'cặp nghệ sĩ thị trường ({len(df_mart_artist_chart):,}):')
display(df_mart_artist_chart.head(3))

cặp nghệ sĩ thị trường (50,203):


,artist_id,chart_country,artist_individual,total_streams,track_count,n_weeks,peak_rank,first_chart_week,last_chart_week,streams_credit_policy
0,000TJhsWOqbyNbTqCU26cW,Netherlands,KlikKlak,297007,1,1,69,2022-07-14,2022-07-14,full credit per collaborating artist; non-addi...
1,001oyhl94fb7YKL6fVqHOo,Poland,Opiat,28907060,10,76,9,2021-02-04,2022-07-14,full credit per collaborating artist; non-addi...
2,004cRPgBOiWUUsu3mXXd1W,Pakistan,RAAMIS,9745,1,1,200,2022-05-12,2022-05-12,full credit per collaborating artist; non-addi...


### Kiểm tra thời gian quan sát của từng thị trường
Tạo danh sách tuần từ đầu đến cuối dữ liệu của mỗi thị trường. Global và các quốc gia có ít nhất 75 tuần được chọn cho tập học chính; các thị trường ngắn hơn vẫn được giữ để phân tích.

In [62]:
df_market_bounds = df_slots_observed.groupby('chart_country', observed=True, as_index=False).agg(
    source_first_week=('week', 'min'), source_last_week=('week', 'max'), observed_weeks=('week', 'nunique'))
df_market_bounds['primary_market'] = df_market_bounds['chart_country'].eq('Global') | df_market_bounds['observed_weeks'].ge(minimum_country_observed_weeks)
assert df_market_bounds['chart_country'].is_unique
df_market_calendar = pd.concat([pd.DataFrame({'chart_country': row.chart_country,
    'week': pd.date_range(row.source_first_week, row.source_last_week, freq='7D')})
    for row in df_market_bounds.itertuples()], ignore_index=True)
assert not df_market_calendar.duplicated(['chart_country', 'week']).any()
assert df_market_calendar['week'].dt.weekday.eq(3).all()
print(f'thị trường ({len(df_market_bounds):,}) và lịch tuần ({len(df_market_calendar):,}):')
display(df_market_bounds.groupby(['primary_market', 'observed_weeks']).size().to_frame('markets'))

thị trường (74) và lịch tuần (5,551):


markets
primary_market observed_weeks         
False          10                    1
               17                    1
               22                    3
True           75                    3
               76                   65
               290                   1

### Đếm thứ hạng và lượt nghe trong từng tuần
Đếm các thứ hạng thực tế và liệt kê vị trí còn thiếu. Tổng lượt nghe chỉ được xem là đủ khi có toàn bộ 200 vị trí và các giá trị lượt nghe đều hợp lệ.

In [63]:
df_market_counts = df_slots_observed.groupby(['chart_country', 'week'], observed=True, as_index=False).agg(
    rank_count=('rank', 'nunique'), streams_count=('streams', 'count'), chart_streams_total=('streams', lambda values: values.sum(min_count=1)))
df_top50_counts = df_slots_observed.loc[df_slots_observed['rank'].le(50)].groupby(['chart_country', 'week'], observed=True)['rank'].nunique().to_frame('top50_rank_count').reset_index()
assert df_slots_observed['rank'].between(1, 200).all()
df_observed_rank_sets = df_slots_observed.groupby(['chart_country', 'week'], observed=True)['rank'].agg(lambda values: set(values)).reset_index(name='observed_rank_set')
assert len(df_observed_rank_sets) == len(df_market_counts)
df_observed_rank_sets['missing_ranks'] = df_observed_rank_sets['observed_rank_set'].map(lambda ranks: sorted(set(range(1, 201)) - ranks))
df_observed_rank_sets['missing_top50_ranks'] = df_observed_rank_sets['observed_rank_set'].map(lambda ranks: sorted(set(range(1, 51)) - ranks))
print(f'tuần có quan sát ({len(df_market_counts):,}):')

tuần có quan sát (5,548):


### Kiểm tra tuần thiếu dữ liệu
Ghép số thứ hạng quan sát được vào danh sách tuần của từng thị trường. Nhờ đó có thể phân biệt tuần mất toàn bộ dữ liệu với tuần chỉ thiếu một số vị trí.

In [64]:
df_market_week_quality = df_market_calendar.merge(df_market_counts, on=['chart_country', 'week'], how='left', validate='one_to_one', indicator='calendar_join')
assert len(df_market_week_quality) == len(df_market_calendar)
df_market_week_quality = df_market_week_quality.merge(df_top50_counts, on=['chart_country', 'week'], how='left', validate='one_to_one')
assert len(df_market_week_quality) == len(df_market_calendar)
df_market_week_quality['chart_observed'] = df_market_week_quality['calendar_join'].eq('both')
df_market_week_quality[['rank_count', 'streams_count', 'top50_rank_count']] = df_market_week_quality[['rank_count', 'streams_count', 'top50_rank_count']].fillna(0).astype('int16')
df_market_week_quality['top200_complete'] = df_market_week_quality['rank_count'].eq(200)
df_market_week_quality['top50_complete'] = df_market_week_quality['top50_rank_count'].eq(50)
df_market_week_quality['streams_total_complete'] = df_market_week_quality['top200_complete'] & df_market_week_quality['streams_count'].eq(200)
assert not df_market_week_quality.duplicated(['chart_country', 'week']).any()
print(f'calendar trước và sau ghép ({len(df_market_calendar):,} → {len(df_market_week_quality):,}):')

calendar trước và sau ghép (5,551 → 5,551):


### Lưu danh sách vị trí xếp hạng bị thiếu
Bổ sung danh sách vị trí thiếu cho cả những tuần không có dòng dữ liệu. Kiểm tra riêng độ đầy đủ của Top 50 để dùng khi tạo nhãn.

In [65]:
df_market_week_quality = df_market_week_quality.merge(df_observed_rank_sets[['chart_country', 'week', 'missing_ranks', 'missing_top50_ranks']],
    on=['chart_country', 'week'], how='left', validate='one_to_one')
assert len(df_market_week_quality) == len(df_market_calendar)
for column, last_rank in [('missing_ranks', 200), ('missing_top50_ranks', 50)]:
    df_market_week_quality[column] = df_market_week_quality[column].map(lambda value: value if isinstance(value, list) else list(range(1, last_rank + 1)))
assert df_market_week_quality['missing_top50_ranks'].map(len).eq(50 - df_market_week_quality['top50_rank_count']).all()
df_missing_market_ranks = df_market_week_quality.loc[~df_market_week_quality['top200_complete']].copy()
assert len(df_missing_market_ranks) == int((~df_market_week_quality['top200_complete']).sum())
print(f"tuần thiếu vị trí ({len(df_missing_market_ranks):,}) và thiếu toàn chart ({int((~df_market_week_quality['chart_observed']).sum()):,}):")
display(df_missing_market_ranks[['chart_country', 'week', 'missing_ranks']].head(10))

tuần thiếu vị trí (79) và thiếu toàn chart (3):


,chart_country,week,missing_ranks
26,Argentina,2021-08-05,[24]
27,Argentina,2021-08-12,[16]
352,Bolivia,2021-08-05,[13]
353,Bolivia,2021-08-12,[11]
656,Chile,2021-08-05,[11]
657,Chile,2021-08-12,[11]
732,Colombia,2021-08-05,[21]
733,Colombia,2021-08-12,[15]
808,Costa Rica,2021-08-05,[23]
809,Costa Rica,2021-08-12,[18]


### Bổ sung mức độ đầy đủ của chart vào bảng tuần
Ghép cờ đủ Top 50, Top 200 và số vị trí bị mất sau khi lọc âm thanh. Các trường này giúp đọc đúng những tuần có dữ liệu chưa đầy đủ.

In [66]:
quality_cols = ['chart_country', 'week', 'rank_count', 'top50_complete', 'top200_complete', 'streams_total_complete']
weekly_rows = len(df_mart_weekly)
df_mart_weekly = df_mart_weekly.merge(df_market_week_quality[quality_cols], on=['chart_country', 'week'], how='left', validate='one_to_one')
assert len(df_mart_weekly) == weekly_rows
df_mart_weekly['audio_slots_removed'] = df_mart_weekly['rank_count'] - df_mart_weekly['track_count']
assert df_mart_weekly['audio_slots_removed'].ge(0).all()
print(f'mart tuần trước và sau ghép chất lượng ({weekly_rows:,} → {len(df_mart_weekly):,}):')

mart tuần trước và sau ghép chất lượng (5,548 → 5,548):


## Phần 8: Tạo dữ liệu dự báo Top 50
Lấy thông tin tuần đầu của mỗi bài trong từng thị trường và kiểm tra tám tuần sau đó. Chia dữ liệu theo thời gian, loại cửa sổ chưa theo dõi xong và các mã bài trùng giữa tập.

### Lấy tuần đầu của bài trong từng thị trường
Giữ dòng sớm nhất của từng cặp bài và thị trường trong lịch sử chart. Thứ hạng, lượt nghe và âm thanh tuần đầu được lấy từ chính dòng này.

In [67]:
# Giữ dòng của chính tuần đầu trong từng thị trường.
df_market_first_observed = df_slots_observed.sort_values('week', kind='stable').drop_duplicates(pair_cols).copy()
assert not df_market_first_observed.duplicated(pair_cols).any()
df_market_first_observed['t0_market'] = df_market_first_observed['week']
df_market_first_observed['rank_first_week'] = df_market_first_observed['rank']
df_market_first_observed['streams_first_week'] = df_market_first_observed['streams']
df_market_first_observed['log_streams_first_week'] = np.log1p(df_market_first_observed['streams_first_week'].astype(float))
df_market_first_observed['duration_min'] = df_market_first_observed['duration'] / 60000
df_market_first_observed = df_market_first_observed.merge(df_market_bounds, on='chart_country', how='left', validate='many_to_one')
assert len(df_market_first_observed) == len(df_slots_observed[pair_cols].drop_duplicates())
print(f'vị trí trước và cặp bài thị trường sau ({len(df_slots_observed):,} → {len(df_market_first_observed):,}):')

vị trí trước và cặp bài thị trường sau (1,109,520 → 103,194):


### Đánh dấu mẫu ở đầu và cuối dữ liệu
Đánh dấu bài đã có ngay tuần đầu của nguồn và bài chưa có đủ tám tuần theo dõi. Thời điểm công bố thực tế chưa biết, nên mốc cộng bốn ngày chỉ là giả định được ghi rõ.

In [68]:
df_market_first_observed['window_start'] = df_market_first_observed['t0_market'] + pd.Timedelta(weeks=1)
df_market_first_observed['window_end'] = df_market_first_observed['t0_market'] + pd.Timedelta(weeks=8)
df_market_first_observed['left_censored'] = df_market_first_observed['t0_market'].eq(df_market_first_observed['source_first_week'])
df_market_first_observed['right_censored'] = df_market_first_observed['window_end'].gt(df_market_first_observed['source_last_week'])
df_market_first_observed['prediction_time'] = pd.Series(pd.NaT, index=df_market_first_observed.index, dtype='datetime64[ns, UTC]')
df_market_first_observed['prediction_time_proxy'] = df_market_first_observed['t0_market'].dt.tz_localize('UTC') + pd.Timedelta(days=publication_lag_days_proxy)
df_market_first_observed['label_ready_time_proxy'] = df_market_first_observed['window_end'].dt.tz_localize('UTC') + pd.Timedelta(days=publication_lag_days_proxy)
df_market_first_observed['time_availability_status'] = 'historical publication unknown; constant lag proxy'
df_market_first_observed['audio_availability_status'] = 'source snapshot at t0 row; collection timestamp unknown; stable recording assumed'
display(df_market_first_observed.groupby('chart_country', observed=True)[['left_censored', 'right_censored']].sum().head())

,left_censored,right_censored
chart_country,,
Argentina,200,91
Australia,200,107
Austria,200,177
Belarus,200,94
Belgium,200,126


### Kiểm tra thông tin lần đầu vào bảng
Đối chiếu hạng tuần trước và số tuần trên chart tại lần đầu quan sát. Thông tin không khớp được lưu để xem lại, không dùng để tự sửa tuần đầu.

In [69]:
df_market_first_observed['previous_rank_first_week'] = pd.to_numeric(df_market_first_observed['previous_rank'], errors='coerce')
df_market_first_observed['weeks_on_chart_first_week'] = pd.to_numeric(df_market_first_observed['weeks_on_chart'], errors='coerce')
df_market_first_observed['entry_history_uncertain'] = (df_market_first_observed['previous_rank_first_week'].ge(1)
    | df_market_first_observed['weeks_on_chart_first_week'].gt(1)
    | df_market_first_observed['previous_rank_first_week'].isna() | df_market_first_observed['weeks_on_chart_first_week'].isna())
df_entry_inconsistencies = df_market_first_observed.loc[df_market_first_observed['entry_history_uncertain']].copy()
assert len(df_entry_inconsistencies) == int(df_market_first_observed['entry_history_uncertain'].sum())
print(f'cặp có lịch sử chưa chắc chắn ({len(df_entry_inconsistencies):,}):')

cặp có lịch sử chưa chắc chắn (32,004):


### Kiểm tra các biến đầu vào tuần đầu
Kiểm tra âm thanh và lượt nghe của đúng tuần đầu có hợp lệ hay không. Hai cột key và mode không nằm trong bộ biến dự báo chính của bài toán này.

In [70]:
df_market_first_observed['has_valid_first_week_audio'] = (np.isfinite(df_market_first_observed[forecast_audio_cols]).all(axis=1)
    & df_market_first_observed[unit_audio_cols].ge(0).all(axis=1) & df_market_first_observed[unit_audio_cols].le(1).all(axis=1)
    & df_market_first_observed['duration'].gt(0) & df_market_first_observed['tempo'].ge(0))
df_market_first_observed['has_valid_first_week_streams'] = df_market_first_observed['streams_first_week'].notna()
df_market_first_observed['has_valid_features'] = df_market_first_observed['has_valid_first_week_audio'] & df_market_first_observed['has_valid_first_week_streams']
df_first_week_feature_errors = df_market_first_observed.loc[~df_market_first_observed['has_valid_features']].copy()
assert len(df_first_week_feature_errors) == int((~df_market_first_observed['has_valid_features']).sum())
print(f'cặp thiếu hoặc lỗi x tuần đầu ({len(df_first_week_feature_errors):,}):')

cặp thiếu hoặc lỗi x tuần đầu (16):


### Tính tỷ trọng lượt nghe tuần đầu
Chia lượt nghe của bài cho tổng lượt nghe cùng thị trường và tuần. Chỉ tính khi bảng đủ Top 200 và mẫu số hợp lệ; biến này được giữ để thử thêm ở bước mô hình.

In [71]:
df_first_week_context = df_market_week_quality[['chart_country', 'week', 'chart_streams_total', 'top200_complete', 'streams_total_complete']].copy()
# Giữ dòng của chính tuần đầu trong từng thị trường.
df_market_first_observed = df_market_first_observed.merge(df_first_week_context, on=['chart_country', 'week'], how='left', validate='many_to_one')
assert len(df_market_first_observed) == df_slots_observed[pair_cols].drop_duplicates().shape[0]
df_market_first_observed['chart_streams_total_first_week'] = df_market_first_observed['chart_streams_total']
valid_share = df_market_first_observed['streams_total_complete'] & df_market_first_observed['chart_streams_total_first_week'].gt(0)
df_market_first_observed['stream_share_first_week'] = (df_market_first_observed['streams_first_week']
    / df_market_first_observed['chart_streams_total_first_week']).where(valid_share)
assert df_market_first_observed['stream_share_first_week'].dropna().between(0, 1).all()
print(f'cặp trước và sau ghép ngữ cảnh ({len(df_market_first_observed):,} → {len(df_market_first_observed):,}):')

cặp trước và sau ghép ngữ cảnh (103,194 → 103,194):


### Tạo hàm theo dõi tám tuần tiếp theo
Tạo tám tuần liên tiếp sau tuần đầu và ghép thứ hạng của bài trong cùng thị trường. Hàm cũng kiểm tra tuần đó có đủ dữ liệu Top 50 hay không.

In [72]:
def make_followup_grid(df_first, df_observations, df_quality):
    df_grid = df_first[pair_cols + ['t0_market']].merge(pd.DataFrame({'horizon_week': range(1, 9)}), how='cross')
    assert len(df_grid) == len(df_first) * 8
    df_grid['future_week'] = df_grid['t0_market'] + pd.to_timedelta(df_grid['horizon_week'] * 7, unit='D')
    df_ranks = df_observations[pair_cols + ['week', 'rank']].copy()
    df_ranks['future_week'], df_ranks['rank_future'] = df_ranks['week'], df_ranks['rank']
    df_grid = df_grid.merge(df_ranks[pair_cols + ['future_week', 'rank_future']], on=pair_cols + ['future_week'], how='left', validate='one_to_one')
    assert len(df_grid) == len(df_first) * 8
    df_quality_keys = df_quality[['chart_country', 'week', 'top50_complete', 'chart_observed']].copy()
    df_quality_keys['future_week'] = df_quality_keys['week']
    df_grid = df_grid.merge(df_quality_keys[['chart_country', 'future_week', 'top50_complete', 'chart_observed']], on=['chart_country', 'future_week'], how='left', validate='many_to_one', indicator='future_quality_join')
    assert len(df_grid) == len(df_first) * 8
    df_grid['within_source_calendar'] = df_grid['future_quality_join'].eq('both')
    df_grid[['top50_complete', 'chart_observed']] = df_grid[['top50_complete', 'chart_observed']].fillna(False).astype(bool)
    df_grid['observed_future_hit'] = df_grid['rank_future'].between(1, 50).fillna(False)
    assert df_grid['future_week'].gt(df_grid['t0_market']).all()
    return df_grid
print(f'hàm lưới tám tuần đã khai báo:')

hàm lưới tám tuần đã khai báo:


### Tạo bảng theo dõi cho tất cả mẫu
Áp dụng hàm theo dõi cho mọi cặp bài và thị trường. Mỗi mẫu phải có đúng tám dòng tương ứng với tám tuần theo lịch.

In [73]:
df_market_followup_grid = make_followup_grid(df_market_first_observed, df_slots_observed, df_market_week_quality)
assert not df_market_followup_grid.duplicated(pair_cols + ['horizon_week']).any()
assert df_market_followup_grid.groupby(pair_cols, observed=True)['horizon_week'].nunique().eq(8).all()
assert (df_market_followup_grid['future_week'] - df_market_followup_grid['t0_market']).dt.days.eq(df_market_followup_grid['horizon_week'] * 7).all()
print(f'cặp ({len(df_market_first_observed):,}) và hàng theo dõi ({len(df_market_followup_grid):,}):')

cặp (103,194) và hàng theo dõi (825,552):


### Tổng hợp kết quả tám tuần theo dõi
Ghi nhận bài có vào Top 50 và cửa sổ theo dõi có đầy đủ hay không. Nếu còn tuần thiếu dữ liệu, mẫu chưa được dùng cho tập học chính dù đã thấy bài vào Top 50.

In [74]:
df_label_evidence = df_market_followup_grid.groupby(pair_cols, observed=True, as_index=False).agg(
    followup_rows=('horizon_week', 'size'), observed_future_hit=('observed_future_hit', 'any'),
    top50_complete_weeks=('top50_complete', 'sum'), source_calendar_weeks=('within_source_calendar', 'sum'),
    observed_chart_weeks=('chart_observed', 'sum'))
df_label_evidence['window_complete'] = df_label_evidence['top50_complete_weeks'].eq(8) & df_label_evidence['source_calendar_weeks'].eq(8)
assert df_label_evidence['followup_rows'].eq(8).all()
df_ml = df_market_first_observed.merge(df_label_evidence, on=pair_cols, how='left', validate='one_to_one')
assert len(df_ml) == len(df_market_first_observed)
print(f'cặp trước và sau ghép bằng chứng ({len(df_market_first_observed):,} → {len(df_ml):,}):')

cặp trước và sau ghép bằng chứng (103,194 → 103,194):


### Tạo nhãn dự báo Top 50
Gán nhãn 1 nếu bài vào Top 50 trong tám tuần tiếp theo, nhãn 0 nếu không vào và đã theo dõi đủ. Mẫu ở đầu, cuối nguồn hoặc thiếu tuần được giữ nhãn trống.

In [75]:
df_ml['label_eligible'] = ~df_ml['left_censored'] & ~df_ml['right_censored'] & df_ml['window_complete']
# Mẫu chưa đủ điều kiện theo dõi giữ nhãn trống.
df_ml['hit_next_8_weeks'] = df_ml['observed_future_hit'].astype('Int8').where(df_ml['label_eligible'])
df_ml['label_status'] = np.select([df_ml['left_censored'], df_ml['right_censored'], ~df_ml['window_complete']],
    ['left_censored', 'right_censored', 'incomplete_top50_window'], default='valid')
df_ml['training_eligible'] = df_ml['label_eligible'] & df_ml['primary_market'] & df_ml['has_valid_features']
df_ml['positive_evidence_in_incomplete_window'] = df_ml['observed_future_hit'] & ~df_ml['window_complete']
assert df_ml.loc[~df_ml['label_eligible'], 'hit_next_8_weeks'].isna().all()
assert df_ml.loc[df_ml['hit_next_8_weeks'].eq(0).fillna(False), 'window_complete'].all()
assert df_ml.loc[df_ml['hit_next_8_weeks'].eq(1).fillna(False), 'observed_future_hit'].all()
assert 'pop_score' not in df_ml and 'popularity' not in df_ml
display(df_ml.groupby(['primary_market', 'label_status'], observed=True).size().to_frame('pairs'))

pairs
primary_market label_status                  
False          left_censored             1000
               right_censored             613
               valid                      826
True           incomplete_top50_window   2386
               left_censored            13796
               right_censored            8432
               valid                    76141

### Tách mẫu đủ điều kiện và mẫu đang chờ
Tách dữ liệu thành mẫu đủ điều kiện học, mẫu chưa hết thời gian theo dõi và mẫu bị loại. Ba nhóm phải không chồng nhau và cộng lại bằng toàn bộ số mẫu.

In [76]:
pending_mask = df_ml['right_censored'] & ~df_ml['left_censored'] & df_ml['primary_market'] & df_ml['has_valid_features']
df_ml_labeled = df_ml.loc[df_ml['training_eligible']].copy()
df_ml_pending = df_ml.loc[pending_mask].copy()
df_ml_excluded = df_ml.loc[~df_ml['training_eligible'] & ~pending_mask].copy()
assert len(df_ml_labeled) == int(df_ml['training_eligible'].sum())
assert len(df_ml_pending) == int(pending_mask.sum())
assert len(df_ml_labeled) + len(df_ml_pending) + len(df_ml_excluded) == len(df_ml)
assert df_ml_pending['hit_next_8_weeks'].isna().all()
df_cohort_counts = pd.DataFrame({'cohort': ['all', 'labeled', 'pending', 'excluded'],
    'rows': [len(df_ml), len(df_ml_labeled), len(df_ml_pending), len(df_ml_excluded)]})
display(df_cohort_counts)

,cohort,rows
0,all,103194
1,labeled,76126
2,pending,8431
3,excluded,18637


### Chọn mốc chia train, validation và test
Chọn hai mốc thời gian gần mức 60% và 80% của các mẫu đủ điều kiện. Mẫu có cùng tuần đầu được đưa vào cùng một khoảng thời gian.

In [77]:
validation_start_t0 = df_ml_labeled['t0_market'].quantile(.6, interpolation='nearest')
test_start_t0 = df_ml_labeled['t0_market'].quantile(.8, interpolation='nearest')
assert validation_start_t0 < test_start_t0
validation_start_proxy = validation_start_t0.tz_localize('UTC') + pd.Timedelta(days=publication_lag_days_proxy)
test_start_proxy = test_start_t0.tz_localize('UTC') + pd.Timedelta(days=publication_lag_days_proxy)
df_temporal_split = df_ml_labeled[pair_cols + ['t0_market', 'window_end', 'label_ready_time_proxy', 'hit_next_8_weeks']].copy()
df_temporal_split['candidate_split'] = np.select([df_temporal_split['t0_market'].lt(validation_start_t0),
    df_temporal_split['t0_market'].lt(test_start_t0)], ['train', 'validation'], default='test')
assert df_temporal_split.groupby('t0_market')['candidate_split'].nunique().eq(1).all()
print(f'bắt đầu validation ({validation_start_t0.date()}) và test ({test_start_t0.date()}):')
display(df_temporal_split.groupby('candidate_split').size().to_frame('before_purge'))

bắt đầu validation (2021-11-04) và test (2022-02-03):


,before_purge
candidate_split,
test,15557
train,44587
validation,15982


### Loại mẫu có nhãn chưa xác định tại mốc chia
Bỏ những mẫu chưa theo dõi xong tám tuần trước khi bắt đầu tập tiếp theo. Cách này tránh dùng kết quả tương lai khi huấn luyện hoặc chọn mô hình.

In [78]:
# Loại mẫu chưa biết đủ kết quả trước khi sang tập tiếp theo.
df_temporal_split['boundary_purged'] = ((df_temporal_split['candidate_split'].eq('train') & df_temporal_split['label_ready_time_proxy'].ge(validation_start_proxy))
    | (df_temporal_split['candidate_split'].eq('validation') & df_temporal_split['label_ready_time_proxy'].ge(test_start_proxy)))
df_temporal_split['split_status'] = np.where(df_temporal_split['boundary_purged'], 'purged_label_window', 'retained')
df_boundary_purged = df_temporal_split.loc[df_temporal_split['boundary_purged']].copy()
assert len(df_boundary_purged) == int(df_temporal_split['boundary_purged'].sum())
print(f'mẫu trước và sau loại cửa sổ chưa hoàn tất mốc chia ({len(df_temporal_split):,} → {len(df_temporal_split) - len(df_boundary_purged):,}):')
display(df_temporal_split.groupby(['candidate_split', 'split_status']).size().to_frame('rows'))

mẫu trước và sau loại cửa sổ chưa hoàn tất mốc chia (76,126 → 58,395):


rows
candidate_split split_status              
test            retained             15557
train           purged_label_window   9135
                retained             35452
validation      purged_label_window   8596
                retained              7386

### Loại bài đã có trong train khỏi validation
Nếu một bài xuất hiện ở nhiều thị trường, bài đó có thể nằm ở nhiều khoảng thời gian. Loại các bài đã có trong train khỏi validation để kiểm tra trên bài chưa gặp.

In [79]:
train_mask = df_temporal_split['candidate_split'].eq('train') & ~df_temporal_split['boundary_purged']
train_track_ids = set(df_temporal_split.loc[train_mask, 'track_id'])
validation_duplicate_mask = (df_temporal_split['candidate_split'].eq('validation') & ~df_temporal_split['boundary_purged']
    & df_temporal_split['track_id'].isin(train_track_ids))
df_temporal_split.loc[validation_duplicate_mask, 'split_status'] = 'excluded_track_seen_in_train'
validation_mask = df_temporal_split['candidate_split'].eq('validation') & df_temporal_split['split_status'].eq('retained')
validation_track_ids = set(df_temporal_split.loc[validation_mask, 'track_id'])
assert not train_track_ids & validation_track_ids
print(f"validation trước và sau loại track đã gặp ({int((df_temporal_split['candidate_split'].eq('validation') & ~df_temporal_split['boundary_purged']).sum()):,} → {int(validation_mask.sum()):,}):")

validation trước và sau loại track đã gặp (7,386 → 6,696):


### Loại bài đã có trong train hoặc validation khỏi test
Giữ trong test các mã bài chưa được dùng ở train và validation. Việc loại dựa trên mã bài và thời gian, không dựa vào nhãn.

In [80]:
test_duplicate_mask = (df_temporal_split['candidate_split'].eq('test')
    & df_temporal_split['track_id'].isin(train_track_ids | validation_track_ids))
df_temporal_split.loc[test_duplicate_mask, 'split_status'] = 'excluded_track_seen_in_train_or_validation'
test_mask = df_temporal_split['candidate_split'].eq('test') & df_temporal_split['split_status'].eq('retained')
test_track_ids = set(df_temporal_split.loc[test_mask, 'track_id'])
assert not test_track_ids & (train_track_ids | validation_track_ids)
df_temporal_split['split'] = df_temporal_split['candidate_split'].where(df_temporal_split['split_status'].eq('retained'), 'excluded')
df_split_exclusions = df_temporal_split.loc[df_temporal_split['split'].eq('excluded')].copy()
assert len(df_split_exclusions) == int(df_temporal_split['split'].eq('excluded').sum())
print(f"test trước và sau loại track đã gặp ({int(df_temporal_split['candidate_split'].eq('test').sum()):,} → {int(test_mask.sum()):,}):")

test trước và sau loại track đã gặp (15,557 → 14,607):


### Tạo ba tập dữ liệu và kiểm tra lại
Ghép thông tin chia tập vào dữ liệu rồi tách train, validation và test. Kiểm tra khóa, thứ tự thời gian, hai lớp nhãn và các biến đầu vào hợp lệ.

In [81]:
df_ml_labeled = df_ml_labeled.merge(df_temporal_split[pair_cols + ['candidate_split', 'split_status', 'split']], on=pair_cols, how='left', validate='one_to_one')
assert len(df_ml_labeled) == len(df_temporal_split)
df_ml_train = df_ml_labeled.loc[df_ml_labeled['split'].eq('train')].copy()
df_ml_validation = df_ml_labeled.loc[df_ml_labeled['split'].eq('validation')].copy()
df_ml_test = df_ml_labeled.loc[df_ml_labeled['split'].eq('test')].copy()
assert len(df_ml_train) + len(df_ml_validation) + len(df_ml_test) + len(df_split_exclusions) == len(df_ml_labeled)
assert df_ml_train['label_ready_time_proxy'].max() < validation_start_proxy
assert df_ml_validation['label_ready_time_proxy'].max() < test_start_proxy
assert df_ml_train['t0_market'].max() < df_ml_validation['t0_market'].min() < df_ml_test['t0_market'].min()
for df_partition in [df_ml_train, df_ml_validation, df_ml_test]:
    assert df_partition['hit_next_8_weeks'].nunique() == 2 and not df_partition.duplicated(pair_cols).any()
    assert np.isfinite(df_partition[[column for column in feature_cols if column != 'chart_country']]).all().all()
print(f'tập mẫu trước và sau ghép split ({len(df_temporal_split):,} → {len(df_ml_labeled):,}):')

tập mẫu trước và sau ghép split (76,126 → 76,126):


### Thống kê số mẫu và tỷ lệ nhãn theo tập
Thống kê số dòng, số bài và tỷ lệ vào Top 50 của từng tập và thị trường. Một bài có nhiều thị trường tạo nhiều mẫu, nên số mẫu khác với số bài độc lập.

In [82]:
df_split_summary = df_ml_labeled.groupby(['split', 'chart_country'], observed=True, as_index=False).agg(
    rows=('track_id', 'size'), tracks=('track_id', 'nunique'), positives=('hit_next_8_weeks', 'sum'),
    first_t0=('t0_market', 'min'), last_t0=('t0_market', 'max'))
df_split_summary['positive_rate'] = df_split_summary['positives'] / df_split_summary['rows']
df_split_overview = df_ml_labeled.groupby('split', observed=True, as_index=False).agg(
    rows=('track_id', 'size'), tracks=('track_id', 'nunique'), positives=('hit_next_8_weeks', 'sum'))
df_split_overview['positive_rate'] = df_split_overview['positives'] / df_split_overview['rows']
display(df_split_overview)

,split,rows,tracks,positives,positive_rate
0,excluded,19371,7143,4503,0.232461
1,test,14607,6222,3740,0.256042
2,train,35452,15665,8386,0.236545
3,validation,6696,2498,1582,0.23626


### Kiểm tra nghệ sĩ xuất hiện ở nhiều tập
Đếm nghệ sĩ chung giữa train, validation và test. Bài toán hiện chia theo mã bài và thời gian; kết quả này giúp thấy phần nghệ sĩ còn giao nhau.

In [83]:
artist_sets = {}
artist_audit = []
for split_name, df_partition in [('train', df_ml_train), ('validation', df_ml_validation), ('test', df_ml_test)]:
    artist_mask = df_track_artist_bridge['track_id'].isin(df_partition['track_id'])
    df_artist_subset = df_track_artist_bridge.loc[artist_mask]
    assert len(df_artist_subset) == int(artist_mask.sum())
    artist_sets[split_name] = set(df_artist_subset['artist_id'])
for left, right in [('train', 'validation'), ('train', 'test'), ('validation', 'test')]:
    artist_audit.append({'left': left, 'right': right, 'overlap_artists': len(artist_sets[left] & artist_sets[right]),
                         'policy': 'diagnostic only; primary split is temporal and track-disjoint'})
df_artist_overlap_audit = pd.DataFrame(artist_audit)
display(df_artist_overlap_audit)

,left,right,overlap_artists,policy
0,train,validation,1299,diagnostic only; primary split is temporal and...
1,train,test,2339,diagnostic only; primary split is temporal and...
2,validation,test,1012,diagnostic only; primary split is temporal and...


### Kiểm tra các mã bài có thể là cùng bản thu
Tìm những mã bài có tên, nghệ sĩ và thời lượng gần giống nhau. Đây là danh sách cần xem lại, chưa đủ để tự gộp các mã bài.

In [84]:
df_recording_review = df_market_first_observed[['track_id', 'track_name', 'artist_names', 'duration']].drop_duplicates('track_id').copy()
assert df_recording_review['track_id'].is_unique
df_recording_review['name_normalized'] = df_recording_review['track_name'].astype('str').str.casefold().str.replace(r'\W+', ' ', regex=True).str.strip()
df_recording_review['artists_normalized'] = df_recording_review['artist_names'].astype('str').str.casefold().str.replace(r'\W+', ' ', regex=True).str.strip()
df_recording_review['duration_second'] = (df_recording_review['duration'] / 1000).round()
recording_keys = ['name_normalized', 'artists_normalized', 'duration_second']
review_mask = df_recording_review.duplicated(recording_keys, keep=False)
df_duplicate_recording_review = df_recording_review.loc[review_mask].copy()
assert len(df_duplicate_recording_review) == int(review_mask.sum())
df_track_split_sets = df_temporal_split.loc[df_temporal_split['split'].ne('excluded')].groupby('track_id')['split'].agg(lambda values: ','.join(sorted(set(values)))).to_frame('retained_split').reset_index()
df_duplicate_recording_review = df_duplicate_recording_review.merge(df_track_split_sets, on='track_id', how='left', validate='one_to_one')
assert len(df_duplicate_recording_review) == int(review_mask.sum())
print(f'id cần rà soát phiên bản ({len(df_duplicate_recording_review):,}):')

id cần rà soát phiên bản (7,530):


### Kiểm tra popularity trong file songs
Kiểm tra mã bài, giới hạn popularity và số bài chart khớp với file songs. Popularity này chưa có thời điểm đo lịch sử nên không được dùng làm đầu vào hay nhãn dự báo.

In [85]:
df_songs_snapshot = pd.read_csv(path_songs, usecols=['id', 'popularity'])
assert df_songs_snapshot['id'].notna().all() and df_songs_snapshot['id'].is_unique
snapshot_popularity = pd.to_numeric(df_songs_snapshot['popularity'], errors='coerce')
assert snapshot_popularity.notna().all() and snapshot_popularity.between(0, 100).all()
df_snapshot_audit = pd.DataFrame({'metric': ['rows', 'zero_popularity', 'chart_tracks_matched', 'chart_pairs_matched'],
    'value': [len(df_songs_snapshot), int(snapshot_popularity.eq(0).sum()),
              int(df_history_tracks['track_id'].isin(df_songs_snapshot['id']).sum()),
              int(df_market_first_observed['track_id'].isin(df_songs_snapshot['id']).sum())]})
display(df_snapshot_audit)

,metric,value
0,rows,550622
1,zero_popularity,148778
2,chart_tracks_matched,3473
3,chart_pairs_matched,20710


### Tạo dữ liệu nhỏ để kiểm tra cách gán nhãn
Tạo các ví dụ có bài vào Top 50 ở tuần đầu, thị trường khác, tuần thứ chín hoặc cửa sổ thiếu tuần. Thêm một ví dụ thiếu âm thanh để kiểm tra thứ hạng tương lai vẫn được giữ.

In [86]:
toy_t0 = pd.Timestamp('2021-01-07')
df_toy_first = pd.DataFrame({'track_id': ['t0_only', 'future_hit', 'other_market', 'week_nine', 'missing_week', 'right_edge', 'audio_missing'],
    'chart_country': ['A', 'A', 'A', 'A', 'C', 'D', 'A'], 't0_market': toy_t0})
df_toy_observations = pd.DataFrame({'track_id': ['t0_only', 'future_hit', 'other_market', 'week_nine', 'audio_missing'],
    'chart_country': ['A', 'A', 'B', 'A', 'A'], 'week': [toy_t0, toy_t0 + pd.Timedelta(weeks=3),
    toy_t0 + pd.Timedelta(weeks=2), toy_t0 + pd.Timedelta(weeks=9), toy_t0 + pd.Timedelta(weeks=1)],
    'rank': [1, 20, 10, 5, 30], 'energy': [0.5, 0.5, 0.5, 0.5, np.nan]})
df_toy_quality = pd.DataFrame([(market, toy_t0 + pd.Timedelta(weeks=week), True, True)
    for market in ['A', 'B', 'C', 'D'] for week in range(1, 9) if not (market == 'D' and week > 3)],
    columns=['chart_country', 'week', 'top50_complete', 'chart_observed'])
missing_toy = df_toy_quality['chart_country'].eq('C') & df_toy_quality['week'].eq(toy_t0 + pd.Timedelta(weeks=4))
df_toy_quality.loc[missing_toy, 'top50_complete'] = False
assert df_toy_quality.loc[missing_toy].shape[0] == 1

### Kiểm tra nhãn trên các ví dụ đã tạo
Chạy hàm theo dõi tám tuần trên các ví dụ nhỏ rồi đối chiếu kết quả mong đợi. Bài vào Top 50 chỉ ở tuần đầu, thị trường khác hoặc tuần thứ chín không được tính là nhãn 1.

In [87]:
df_toy_grid = make_followup_grid(df_toy_first, df_toy_observations, df_toy_quality)
df_toy_results = df_toy_grid.groupby('track_id').agg(hit=('observed_future_hit', 'any'),
    complete_weeks=('top50_complete', 'sum'), calendar_weeks=('within_source_calendar', 'sum'))
assert df_toy_results.loc[['t0_only', 'other_market', 'week_nine'], 'hit'].eq(False).all()
assert df_toy_results.loc[['future_hit', 'audio_missing'], 'hit'].eq(True).all()
assert df_toy_results.loc['missing_week', 'complete_weeks'] == 7
assert df_toy_results.loc['right_edge', 'calendar_weeks'] == 3
assert len(df_toy_grid) == len(df_toy_first) * 8
display(df_toy_results)

,hit,complete_weeks,calendar_weeks
track_id,,,
audio_missing,True,8,8
future_hit,True,8,8
missing_week,False,7,8
other_market,False,8,8
right_edge,False,3,3
t0_only,False,8,8
week_nine,False,8,8


### Đối chiếu nhãn với dữ liệu chart gốc
Lấy 120 mẫu và kiểm tra trực tiếp thứ hạng trong tám tuần sau tuần đầu. Cách kiểm tra này độc lập với bảng tổng hợp dùng để tạo nhãn.

In [88]:
df_label_spotcheck = df_ml_labeled.sample(n=min(120, len(df_ml_labeled)), random_state=42)
assert len(df_label_spotcheck) == min(120, len(df_ml_labeled))
for sample in df_label_spotcheck.itertuples():
    spot_mask = df_slots_observed['track_id'].eq(sample.track_id) & df_slots_observed['chart_country'].eq(sample.chart_country)
    spot_mask &= df_slots_observed['week'].gt(sample.t0_market) & df_slots_observed['week'].le(sample.window_end)
    spot_mask &= df_slots_observed['rank'].le(50)
    assert int(spot_mask.any()) == sample.hit_next_8_weeks
print(f'nhãn đối chiếu độc lập đạt ({len(df_label_spotcheck):,}):')

nhãn đối chiếu độc lập đạt (120):


### Chốt các biến đầu vào và quy tắc chia tập
Lưu danh sách biến tuần đầu, nhãn và các mốc chia dữ liệu để notebook mô hình dùng lại. Những bước học tham số xử lý hoặc chọn ngưỡng sẽ làm trên train trong phần mô hình.

In [89]:
# Không dùng các chỉ số tổng hợp từ tương lai làm biến đầu vào.
blocked_feature_cols = ['hit_next_8_weeks', 'observed_future_hit', 'total_streams', 'nweek', 'peak_rank', 'popularity', 'pop_score']
assert not set(feature_cols).intersection(blocked_feature_cols)
forecast_config = {'grain': pair_cols, 'features': feature_cols, 'optional_features': ['stream_share_first_week'],
    'target': 'hit_next_8_weeks', 'horizon_weeks': 8, 'threshold_rank': 50,
    'validation_start_t0': str(validation_start_t0), 'test_start_t0': str(test_start_t0),
    'publication_lag_days_proxy': publication_lag_days_proxy, 'actual_publication_timestamp_known': False,
    'primary_market_minimum_observed_weeks': minimum_country_observed_weeks,
    'split_policy': 'chronological + label maturity purge + track-disjoint retained sets',
    'audio_as_of_t0_verified': False, 'preprocessing_fit': '03_model train only',
    'source_selection': 'tracks already observed in Top 200; first observed is not verified debut'}
display(pd.DataFrame({'feature': feature_cols, 'availability': 'first observed chart week; audio historical stability assumed'}))

,feature,availability
0,danceability,first observed chart week; audio historical st...
1,energy,first observed chart week; audio historical st...
2,loudness,first observed chart week; audio historical st...
3,speechiness,first observed chart week; audio historical st...
4,acousticness,first observed chart week; audio historical st...
5,instrumentalness,first observed chart week; audio historical st...
6,liveness,first observed chart week; audio historical st...
7,valence,first observed chart week; audio historical st...
8,tempo,first observed chart week; audio historical st...
9,duration_min,first observed chart week; audio historical st...


## Phần 9: Kiểm tra và xuất dữ liệu
Chuẩn bị từ điển dữ liệu, bảng tóm tắt quy trình và các file đầu ra. Đọc lại Parquet để đối chiếu trước khi tổng kết số mẫu.

### Chuẩn bị các bảng để xuất file
Gom các bảng sạch, bảng phân tích và tập dự báo vào danh sách xuất. Tách riêng X và y, rồi kiểm tra chúng có cùng số dòng và khóa mẫu.

In [90]:
main_outputs = {'charts_clean': df_charts_clean, 'chart_slots': df_slots, 'chart_slots_observed': df_slots_observed,
    'tracks_dimension': df_tracks, 'track_artist_bridge': df_track_artist_bridge,
    'market_week_quality': df_market_week_quality, 'market_first_observed': df_market_first_observed,
    'market_followup_grid': df_market_followup_grid, 'ml_market_first_week': df_ml,
    'ml_market_labeled': df_ml_labeled, 'ml_market_pending': df_ml_pending, 'ml_market_excluded': df_ml_excluded,
    'ml_market_train': df_ml_train, 'ml_market_validation': df_ml_validation, 'ml_market_test': df_ml_test,
    'track_week_performance_descriptive': df_track_week_performance, 'track_performance_descriptive': df_track_performance,
    'mart_audio_trend_weekly': df_mart_weekly, 'mart_country_year': df_mart_country_year,
    'mart_artist_chart': df_mart_artist_chart, 'country_lookup': df_country_lookup, 'covid_weekly': df_covid_weekly_clean}
for split_name, df_split in [('train', df_ml_train), ('validation', df_ml_validation), ('test', df_ml_test)]:
    matrix_cols = list(dict.fromkeys(pair_cols + feature_cols))
    main_outputs['X_' + split_name] = df_split[matrix_cols].copy()
    main_outputs['y_' + split_name] = df_split[pair_cols + ['hit_next_8_weeks']].copy()
    assert len(main_outputs['X_' + split_name]) == len(main_outputs['y_' + split_name])
print(f'bảng chính và ma trận đã chuẩn bị ({len(main_outputs):,}):')

bảng chính và ma trận đã chuẩn bị (28):


### Tạo từ điển dữ liệu
Ghi tên cột, kiểu dữ liệu, vai trò và số giá trị thiếu của từng bảng. Các cột dùng làm nhãn hoặc kiểm tra kết quả tương lai được phân biệt với biến đầu vào.

In [91]:
dictionary_rows = []
forecast_tables = {name for name in main_outputs if name.startswith(('ml_market_', 'X_', 'y_'))}
future_cols = {'observed_future_hit', 'complete_top50_weeks', 'within_calendar_weeks', 'observed_chart_weeks', 'window_complete', 'positive_evidence_in_incomplete_window'}
for table_name, df_output in main_outputs.items():
    for column in df_output.columns:
        role = 'id' if column in pair_cols else 'provenance'
        if table_name in forecast_tables and column in feature_cols:
            role = 'feature'
        if column == 'stream_share_first_week' and table_name in forecast_tables:
            role = 'optional_feature'
        if column == 'hit_next_8_weeks':
            role = 'outcome'
        if column in future_cols:
            role = 'outcome_audit'
        availability = 'future; excluded from X' if role.startswith('outcome') else ('t0; audio stability assumed' if 'feature' in role else 'metadata/descriptive; not automatically usable in X')
        dictionary_rows.append({'table': table_name, 'column': column, 'dtype': str(df_output[column].dtype),
            'role': role, 'availability': availability, 'missing_count': int(df_output[column].isna().sum())})
df_data_dictionary = pd.DataFrame(dictionary_rows)
display(df_data_dictionary.loc[df_data_dictionary['table'].eq('X_train')])
assert len(df_data_dictionary) == sum(len(df_output.columns) for df_output in main_outputs.values())

,table,column,dtype,role,availability,missing_count
971,X_train,track_id,object,id,metadata/descriptive; not automatically usable...,0
972,X_train,chart_country,str,feature,t0; audio stability assumed,0
973,X_train,danceability,float64,feature,t0; audio stability assumed,0
974,X_train,energy,float64,feature,t0; audio stability assumed,0
975,X_train,loudness,float64,feature,t0; audio stability assumed,0
976,X_train,speechiness,float64,feature,t0; audio stability assumed,0
977,X_train,acousticness,float64,feature,t0; audio stability assumed,0
978,X_train,instrumentalness,float64,feature,t0; audio stability assumed,0
979,X_train,liveness,float64,feature,t0; audio stability assumed,0
980,X_train,valence,float64,feature,t0; audio stability assumed,0


### Bổ sung khóa và vai trò các cột trong từ điển
Ghi các cột xác định một dòng của từng bảng và đánh dấu đầy đủ những trường kiểm tra tương lai. Các trường này giúp đối chiếu kết quả nhưng không nằm trong X.

In [92]:
grain_map = {'charts_clean': 'track_id, artist_id, chart_country, week', 'chart_slots': 'chart_country, week, rank',
    'chart_slots_observed': 'chart_country, week, rank', 'tracks_dimension': 'track_id',
    'track_artist_bridge': 'track_id, artist_id', 'market_week_quality': 'chart_country, week',
    'market_followup_grid': 'track_id, chart_country, horizon_week', 'country_lookup': 'chart_country',
    'mart_audio_trend_weekly': 'chart_country, week', 'mart_country_year': 'chart_country, year',
    'mart_artist_chart': 'artist_id, chart_country', 'covid_weekly': 'country, week',
    'track_week_performance_descriptive': 'track_id, week', 'track_performance_descriptive': 'track_id'}
df_data_dictionary['grain'] = df_data_dictionary['table'].map(grain_map).fillna('track_id, chart_country')
future_fields = ['followup_rows', 'observed_future_hit', 'top50_complete_weeks', 'source_calendar_weeks',
    'observed_chart_weeks', 'window_complete', 'positive_evidence_in_incomplete_window',
    'label_eligible', 'training_eligible', 'label_status', 'right_censored', 'rank_future']
future_mask = df_data_dictionary['column'].isin(future_fields)
df_data_dictionary.loc[future_mask, ['role', 'availability']] = ['outcome_audit', 'future/source coverage; excluded from X']
assert df_data_dictionary['grain'].notna().all()
assert df_data_dictionary.loc[future_mask, 'role'].eq('outcome_audit').all()
print(f'trường từ điển đã rà soát ({len(df_data_dictionary):,}):')

trường từ điển đã rà soát (1,022):


### Bổ sung đơn vị và ý nghĩa các cột
Ghi đơn vị của đặc trưng âm thanh, lượt nghe và nhãn, cùng ý nghĩa các trường chính. Những đơn vị chưa xác minh được ghi rõ trong từ điển.

In [93]:
unit_map = {column: '0–1' for column in unit_audio_cols}
unit_map.update({'loudness': 'dB', 'tempo': 'BPM', 'duration': 'ms', 'duration_min': 'phút',
    'rank': 'vị trí 1–200', 'rank_first_week': 'vị trí 1–200', 'streams': 'lượt nghe trong chart',
    'streams_first_week': 'lượt nghe cùng thị trường tuần đầu', 'total_streams': 'lượt nghe; kiểm tra phạm vi bảng',
    'hit_next_8_weeks': '0/1 hoặc thiếu', 'gdp_source_value': 'chưa xác minh đơn vị snapshot',
    'stream_share_first_week': '0–1; mẫu số chỉ dùng chart đủ Top 200'})
description_map = {'track_id': 'ID Spotify của bài', 'chart_country': 'thị trường chart; Global là thị trường riêng',
    't0_market': 'tuần đầu quan sát trong nguồn của đúng thị trường; không xác nhận debut',
    'hit_next_8_weeks': 'có rank 1–50 trong t0+7 đến t0+56 ngày, chỉ gán khi đủ cửa sổ và không ở biên',
    'rank_first_week': 'hạng ở t0 của chính thị trường', 'log_streams_first_week': 'ln(1 + streams_first_week)',
    'prediction_time': 'timestamp công bố thực tế chưa có bằng chứng, giữ thiếu',
    'prediction_time_proxy': 't0 UTC + 4 ngày, giả định công bố dùng để chia tập',
    'label_ready_time_proxy': 'window_end UTC + 4 ngày, dùng purge theo cùng giả định',
    'stream_scope': 'phạm vi tổng streams ưu tiên Global theo bài-tuần; chỉ mô tả',
    'release_date_precision': 'mức chính xác ngày/tháng/năm của chuỗi ngày phát hành'}
df_data_dictionary['unit'] = df_data_dictionary['column'].map(unit_map).fillna('xem nguồn hoặc cấu hình trường')
df_data_dictionary['description'] = df_data_dictionary['column'].map(description_map).fillna(df_data_dictionary['column'])
assert not df_data_dictionary.duplicated(['table', 'column']).any()

### Tóm tắt các bước xử lý dữ liệu
Tạo bảng tóm tắt đầu vào, đầu ra, khóa và cách kiểm tra của từng bước. Bảng này có thể dùng để trình bày quy trình trong báo cáo.

In [94]:
df_pipeline_steps = pd.DataFrame([
    ('làm sạch chart', 'CSV chart', 'charts_clean', 'bài–nghệ sĩ–thị trường–tuần', 'kiểu, miền giá trị, lý do cách ly, bảo toàn số dòng'),
    ('lịch độc lập audio', 'chart trước lọc audio', 'chart_slots_observed', 'thị trường–tuần–rank', 'track và streams nhất quán, khóa duy nhất'),
    ('calendar thị trường', 'slot quan sát', 'market_week_quality', 'thị trường–tuần', 'hạng thiếu, tuần mất toàn bộ, Top 50/200 đầy đủ'),
    ('lấy tuần đầu', 'lịch quan sát', 'market_first_observed', 'bài–thị trường', 't0 min, X tuần đầu, cờ biên, timestamp proxy'),
    ('theo dõi tương lai', 'tuần đầu và calendar', 'market_followup_grid', 'bài–thị trường–horizon', 'đúng 8 tuần lịch; không tính t0 hoặc thị trường khác'),
    ('gán nhãn', 'bằng chứng cửa sổ', 'ml_market_labeled/pending/excluded', 'bài–thị trường', 'chỉ nhãn chính khi đủ 8 tuần; pending giữ thiếu'),
    ('chia tập', 'cohort đủ điều kiện', 'X/y train, validation, test', 'bài–thị trường', 'thứ tự thời gian, purge 8 tuần, không chung track_id'),
    ('mart EDA', 'slot sạch và GDP/WHR/COVID', 'mart tuần/năm/nghệ sĩ', 'theo từng mart', 'join có validate, khóa duy nhất, đối chiếu streams'),
    ('xuất dữ liệu', 'DataFrame chuẩn hóa', 'Parquet và quality CSV', 'giữ grain', 'đọc lại từng cột, schema, SHA256 nguồn')],
    columns=['step', 'input', 'output', 'grain', 'check'])
assert len(df_pipeline_steps) == 9
display(df_pipeline_steps)

,step,input,output,grain,check
0,làm sạch chart,CSV chart,charts_clean,bài–nghệ sĩ–thị trường–tuần,"kiểu, miền giá trị, lý do cách ly, bảo toàn số..."
1,lịch độc lập audio,chart trước lọc audio,chart_slots_observed,thị trường–tuần–rank,"track và streams nhất quán, khóa duy nhất"
2,calendar thị trường,slot quan sát,market_week_quality,thị trường–tuần,"hạng thiếu, tuần mất toàn bộ, Top 50/200 đầy đủ"
3,lấy tuần đầu,lịch quan sát,market_first_observed,bài–thị trường,"t0 min, X tuần đầu, cờ biên, timestamp proxy"
4,theo dõi tương lai,tuần đầu và calendar,market_followup_grid,bài–thị trường–horizon,đúng 8 tuần lịch; không tính t0 hoặc thị trườn...
5,gán nhãn,bằng chứng cửa sổ,ml_market_labeled/pending/excluded,bài–thị trường,chỉ nhãn chính khi đủ 8 tuần; pending giữ thiếu
6,chia tập,cohort đủ điều kiện,"X/y train, validation, test",bài–thị trường,"thứ tự thời gian, purge 8 tuần, không chung tr..."
7,mart EDA,slot sạch và GDP/WHR/COVID,mart tuần/năm/nghệ sĩ,theo từng mart,"join có validate, khóa duy nhất, đối chiếu str..."
8,xuất dữ liệu,DataFrame chuẩn hóa,Parquet và quality CSV,giữ grain,"đọc lại từng cột, schema, SHA256 nguồn"


### Chuẩn bị các bảng kiểm tra dữ liệu
Gom các dòng lỗi, trường hợp không khớp, thống kê nhãn và kết quả chia tập để lưu lại. Những bảng này giúp giải thích số mẫu được giữ hoặc loại.

In [95]:
quality_outputs = {'pipeline_steps': df_pipeline_steps, 'numeric_conversion': df_numeric_audit, 'chart_rejected': df_charts_rejected,
    'chart_review': df_chart_review, 'artist_count_exceptions': df_artist_exceptions,
    'history_invalid_rows': df_history_invalid_rows, 'history_stream_errors': df_history_stream_errors,
    'history_slot_conflicts': df_history_slot_conflicts, 'history_conflict_rows': df_history_conflict_rows,
    'audio_lost_tracks': df_audio_lost_tracks, 'missing_market_ranks': df_missing_market_ranks,
    'entry_inconsistencies': df_entry_inconsistencies, 'first_week_feature_errors': df_first_week_feature_errors,
    'cohort_counts': df_cohort_counts, 'boundary_purged': df_boundary_purged, 'split_exclusions': df_split_exclusions,
    'split_summary': df_split_summary, 'split_overview': df_split_overview, 'artist_overlap': df_artist_overlap_audit,
    'duplicate_recording_review': df_duplicate_recording_review, 'streams_audit': df_streams_audit,
    'whr_rejected': df_whs_rejected, 'gdp_unmatched': df_gdp_unmatched, 'whr_unmatched': df_whr_unmatched,
    'covid_unmatched': df_covid_unmatched, 'snapshot_audit': df_snapshot_audit,
    'label_edge_tests': df_toy_results.reset_index(), 'data_dictionary': df_data_dictionary}
print(f'bảng bằng chứng đã chuẩn bị ({len(quality_outputs):,}):')

bảng bằng chứng đã chuẩn bị (28):


### Xuất Parquet và kiểm tra dữ liệu đọc lại
Ghi từng bảng ra Parquet rồi đọc lại từng cột để đối chiếu giá trị, số dòng và tên cột. Đọc theo cột giúp hạn chế bộ nhớ khi kiểm tra bảng lớn.

In [96]:
import pyarrow.parquet as pq
path_quality = path_processed / 'quality'
path_quality.mkdir(parents=True, exist_ok=True)
output_audit_rows = []
for table_name, df_output in main_outputs.items():
    destination = path_processed / f'{table_name}.parquet'
    df_output.to_parquet(destination, index=False)
    parquet_file = pq.ParquetFile(destination)
    assert parquet_file.metadata.num_rows == len(df_output)
    assert parquet_file.schema_arrow.names == list(df_output.columns)
    for column in df_output.columns:
        df_readback = pd.read_parquet(destination, columns=[column])
        pd.testing.assert_series_equal(df_output[column].reset_index(drop=True).convert_dtypes(), df_readback[column].convert_dtypes(), check_dtype=False)
        del df_readback
    gc.collect()
    output_audit_rows.append({'table': table_name, 'rows': len(df_output), 'columns': len(df_output.columns), 'roundtrip_verified': True})
df_output_quality = pd.DataFrame(output_audit_rows)
display(df_output_quality)

,table,rows,columns,roundtrip_verified
0,charts_clean,1787531,45,True
1,chart_slots,1109375,45,True
2,chart_slots_observed,1109520,37,True
3,tracks_dimension,39190,25,True
4,track_artist_bridge,62303,4,True
5,market_week_quality,5551,13,True
6,market_first_observed,103194,66,True
7,market_followup_grid,825552,11,True
8,ml_market_first_week,103194,77,True
9,ml_market_labeled,76126,80,True


### Xuất báo cáo kiểm tra và cấu hình
Lưu các bảng kiểm tra thành CSV cùng cấu hình dự báo và phiên bản thư viện. Các mẫu đang chờ đủ tám tuần tiếp tục giữ nhãn trống.

In [97]:
for table_name, df_output in quality_outputs.items():
    df_output.to_csv(path_quality / f'{table_name}.csv', index=False, encoding='utf-8-sig')
df_output_quality.to_csv(path_quality / 'output_quality.csv', index=False, encoding='utf-8-sig')
for file_name, payload in [('forecast_config.json', forecast_config), ('runtime_versions.json', runtime_versions)]:
    (path_processed / file_name).write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'báo cáo chất lượng đã lưu ({len(quality_outputs) + 1:,}):')

báo cáo chất lượng đã lưu (29):


### Lưu thông tin nhận diện file nguồn
Tính mã SHA256 cho từng file để nhận diện bộ dữ liệu đã dùng. Thông tin nguồn chưa xác minh vẫn để trống thay vì tự suy đoán.

In [98]:
for file_name, source_info in source_manifest.items():
    source_path = path_raw / file_name
    source_hash = hashlib.sha256()
    with source_path.open('rb') as source_file:
        for block in iter(lambda: source_file.read(1024 * 1024), b''):
            source_hash.update(block)
    source_info.update(file=str(source_path), bytes=source_path.stat().st_size, sha256=source_hash.hexdigest())
source_manifest['spotify-weekly-charts.csv'].update(first_week=str(df_slots_observed['week'].min()), last_week=str(df_slots_observed['week'].max()))
(path_processed / 'source_manifest.json').write_text(json.dumps(source_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'file nguồn đã ghi sha256 ({len(source_manifest):,}):')

file nguồn đã ghi sha256 (6):


### Ghi hướng dẫn sử dụng dữ liệu
Lưu cách đọc các bảng, định nghĩa nhãn và giới hạn của bộ dữ liệu. Hướng dẫn cũng chỉ rõ các file X, y dùng trong bước huấn luyện.

In [99]:
readme_lines = ['# Tiền xử lý Top 50 theo thị trường trong 8 tuần',
    'Notebook: notebooks/01_preprocess_market_forecast.ipynb. Các nhánh cũ được giữ để đối chiếu.',
    f'Grain: track_id + chart_country. Tổng mẫu {len(df_ml):,}; đủ điều kiện học {len(df_ml_labeled):,}.',
    f'Train {len(df_ml_train):,}; validation {len(df_ml_validation):,}; test {len(df_ml_test):,}; pending {len(df_ml_pending):,}.',
    'Nhãn: có rank <= 50 ở cùng thị trường trong t0+7 đến t0+56 ngày; t0 không tính.',
    'Chỉ gán nhãn chính khi đủ cả 8 tuần Top 50 và không chạm biên nguồn. Global là một thị trường riêng.',
    'First observed không phải debut đã xác minh. Dữ liệu chỉ đại diện bài từng vào Top 200.',
    'prediction_time chưa biết; proxy t0+4 ngày là giả định công bố, không phải timestamp nguồn.',
    'Audio snapshot được giả định ổn định; chưa xác minh có sẵn tại t0. Popularity snapshot không vào X/y.',
    'X_*/y_* có cùng thứ tự và khóa. fit imputer/scaler/encoder chỉ trên train ở notebook 03_model.',
    'Chia theo thời gian, purge cửa sổ nhãn và giữ track_id tách biệt. Artist overlap chỉ được audit.',
    'Bản thu khác ID có thể còn chồng lấn; quality/duplicate_recording_review.csv là danh sách rà soát.',
    'EDA: mart_audio_trend_weekly, mart_country_year, mart_artist_chart. COVID/WHR/GDP không thuộc X mặc định.',
    'Streams theo thị trường không cộng Global với quốc gia. Tổng mô tả cấp bài ưu tiên Global trong mỗi bài-tuần.',
    'Nguồn, SHA256, điểm chưa xác minh: source_manifest.json. Không tự gán GDP unit hay giấy phép.',
    '```mermaid', 'flowchart LR', 'A[Raw chart] --> B[History slots and calendar] --> C[First observed and future windows] --> D[Labels and temporal split]',
    'A --> E[Audio clean and artist dedup] --> F[EDA marts]; D --> G[X/y and quality audit]', '```']
(path_processed / 'README.md').write_text('\n\n'.join(readme_lines), encoding='utf-8')

1596

### Kiểm tra cuối và tổng kết kết quả
Kiểm tra các tập không chung mã bài, mẫu chờ giữ nhãn trống và mọi bảng đã được đọc lại thành công. In số mẫu cuối để dễ đối chiếu khi làm mô hình.

In [100]:
assert df_ml_train['track_id'].nunique() >= 1 and df_ml_validation['track_id'].nunique() >= 1 and df_ml_test['track_id'].nunique() >= 1
assert not set(df_ml_train['track_id']).intersection(df_ml_validation['track_id'])
assert not set(df_ml_train['track_id']).intersection(df_ml_test['track_id'])
assert not set(df_ml_validation['track_id']).intersection(df_ml_test['track_id'])
assert df_ml_pending['hit_next_8_weeks'].isna().all()
assert df_output_quality['roundtrip_verified'].all()
print(f'mẫu bài thị trường ({len(df_ml):,}), nhãn đủ điều kiện ({len(df_ml_labeled):,}), mẫu đang chờ ({len(df_ml_pending):,}):')
print(f'train ({len(df_ml_train):,}), validation ({len(df_ml_validation):,}), test ({len(df_ml_test):,}):')
display(df_split_overview)

mẫu bài thị trường (103,194), nhãn đủ điều kiện (76,126), mẫu đang chờ (8,431):
train (35,452), validation (6,696), test (14,607):


,split,rows,tracks,positives,positive_rate
0,excluded,19371,7143,4503,0.232461
1,test,14607,6222,3740,0.256042
2,train,35452,15665,8386,0.236545
3,validation,6696,2498,1582,0.23626
